<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/17_Ablation_Study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
# =============================================================================
# NOTEBOOK 17 — ABLATION STUDY
# Research: SPP-GAN
# Statistical-Guided Privacy-Preserving Generative Adversarial Network
#
# Research Objective:
# Evaluate the individual contributions of statistical guidance
# and differential privacy within the proposed SPP-GAN framework.
#
# Ablation Variants:
#   1. GAN-only
#   2. Statistical Guidance + GAN
#   3. GAN + Differential Privacy
#   4. Complete SPP-GAN
#
# Datasets:
#   1. adult_income
#   2. bank_marketing
#   3. diabetes_130us
#
# Experimental Configuration:
#   Master Seed       : 2025
#   Repetitions       : 5
#   Repetition Seeds  : 3026–3030
#   Ablation Variants : 4
#   Expected Records : 60
#
# Evaluation Dimensions:
#   1. Statistical Fidelity
#   2. Downstream Utility (TSTR)
#   3. Privacy and Disclosure Risk
#   4. Computational Efficiency
#   5. Component Contribution
#
# Methodological Alignment:
#   Section 4.9 — Ablation Study
#   Section 4.12 — Overall Evaluation Protocol
#   Section 4.13 — Reproducibility and Artifact Management
#
# Execution Policy:
#   - Consume persisted experimental artifacts.
#   - Do not retrain previously trained models.
#   - Preserve upstream datasets and model checkpoints.
#   - Maintain dataset and repetition-level experimental records.
#   - Save outputs, metadata, manifests and validation reports.
#   - Do not fabricate missing experimental evidence.
#
# Expected Experimental Coverage:
#   3 Datasets × 5 Repetitions × 4 Variants = 60 Records
#
# =============================================================================

import os
import gc
import json
import hashlib
import warnings
import time
import random
import platform

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# =============================================================================
# MASTER EXPERIMENT CONFIGURATION
# =============================================================================

MASTER_SEED = 2025

REPETITIONS = [1, 2, 3, 4, 5]

REPETITION_SEEDS = {
    repetition: MASTER_SEED + 1000 + repetition
    for repetition in REPETITIONS
}

DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]

VARIANTS = [
    "gan_only",
    "statistical_guidance_gan",
    "gan_dp",
    "full_sppgan"
]

EXPECTED_RECORDS = (
    len(DATASETS) *
    len(REPETITIONS) *
    len(VARIANTS)
)

# =============================================================================
# REPRODUCIBILITY
# =============================================================================

random.seed(MASTER_SEED)
np.random.seed(MASTER_SEED)

# =============================================================================
# EXPERIMENT INITIALIZATION
# =============================================================================

NOTEBOOK_NAME = "Notebook 17 — Ablation Study"

START_TIME_UTC = datetime.now(
    timezone.utc
).isoformat()

EXECUTION_TIMESTAMP = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")

print("=" * 80)
print(NOTEBOOK_NAME.upper())
print("=" * 80)

print("Research Framework       : SPP-GAN")
print("Datasets                 :", len(DATASETS))
print("Repetitions              :", len(REPETITIONS))
print("Repetition Seeds         :", list(REPETITION_SEEDS.values()))
print("Ablation Variants        :", len(VARIANTS))
print("Expected Experiment Rows :", EXPECTED_RECORDS)
print("Master Seed              :", MASTER_SEED)
print("Python Version           :", platform.python_version())
print("NumPy Version            :", np.__version__)
print("Pandas Version           :", pd.__version__)
print("Execution Timestamp      :", START_TIME_UTC)

print("=" * 80)
print("NOTEBOOK 17 INITIALIZATION COMPLETED")
print("=" * 80)

NOTEBOOK 17 — ABLATION STUDY
Research Framework       : SPP-GAN
Datasets                 : 3
Repetitions              : 5
Repetition Seeds         : [3026, 3027, 3028, 3029, 3030]
Ablation Variants        : 4
Expected Experiment Rows : 60
Master Seed              : 2025
Python Version           : 3.13.16
NumPy Version            : 2.4.6
Pandas Version           : 2.2.3
Execution Timestamp      : 2026-11-10T18:54:26.991736+00:00
NOTEBOOK 17 INITIALIZATION COMPLETED


In [23]:

# =============================================================================
# SECTION 2 — GOOGLE DRIVE, PROJECT PATHS & CONFIGURATION
# Notebook 17 — Ablation Study
# =============================================================================

import os
import json
import csv
from pathlib import Path
from datetime import datetime

import yaml
from google.colab import drive

# -----------------------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# -----------------------------------------------------------------------------

drive.mount("/content/drive", force_remount=False)

# -----------------------------------------------------------------------------
# 2. PROJECT ROOT
# -----------------------------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/SPP_GAN_Research")
MYDRIVE_ROOT = Path("/content/drive/MyDrive")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root does not exist: {PROJECT_ROOT}\n"
        "Verify the Google Drive project folder."
    )

CONFIG_DIR = PROJECT_ROOT / "config"
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
METADATA_DIR = PROJECT_ROOT / "metadata"

# -----------------------------------------------------------------------------
# 3. NOTEBOOK 17 DIRECTORY STRUCTURE
# -----------------------------------------------------------------------------

NB17_ROOT = RESULTS_DIR / "notebooks" / "notebook_17"

NB17_DIRS = {
    "results": NB17_ROOT / "results",
    "tables": NB17_ROOT / "tables",
    "metadata": NB17_ROOT / "metadata",
    "validation": NB17_ROOT / "validation",
    "figures": NB17_ROOT / "figures",
    "manifests": NB17_ROOT / "manifests",
    "logs": NB17_ROOT / "logs",
    "registries": NB17_ROOT / "registries",
}

for directory in NB17_DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

REGISTRY_PATH = (
    NB17_DIRS["registries"] / "ablation_variant_registry.csv"
)

# -----------------------------------------------------------------------------
# 4. DISCOVER NOTEBOOK 00 MASTER CONFIGURATION
# -----------------------------------------------------------------------------

CONFIG_FILENAMES = {
    "sppgan_model_configuration.json",
    "master_config.json",
    "master_config.yaml",
    "master_config.yml",
    "notebook_00_config.json",
    "notebook_00_config.yaml",
    "notebook_00_config.yml",
}

MASTER_CONFIG_PATH = None
CONFIG_SEARCH_RESULTS = []

# Search only within the confirmed project root.
for current_root, directories, filenames in os.walk(PROJECT_ROOT):

    directories[:] = [
        directory for directory in directories
        if directory not in {
            ".git",
            "__pycache__",
            ".ipynb_checkpoints",
            "node_modules",
        }
    ]

    for filename in filenames:
        if filename.lower() in CONFIG_FILENAMES:
            candidate = Path(current_root) / filename
            CONFIG_SEARCH_RESULTS.append(candidate)

# Prefer the previously specified canonical filename if discovered.
if CONFIG_SEARCH_RESULTS:
    preferred_names = [
        "sppgan_model_configuration.json",
        "master_config.json",
        "master_config.yaml",
        "master_config.yml",
        "notebook_00_config.json",
        "notebook_00_config.yaml",
        "notebook_00_config.yml",
    ]

    for preferred_name in preferred_names:
        matches = [
            path for path in CONFIG_SEARCH_RESULTS
            if path.name.lower() == preferred_name
        ]

        if matches:
            MASTER_CONFIG_PATH = matches[0]
            break

# -----------------------------------------------------------------------------
# 5. LOAD MASTER CONFIGURATION WITHOUT ASSUMING ITS LOCATION
# -----------------------------------------------------------------------------

MASTER_CONFIG = {}
MASTER_CONFIG_LOADED = False

if MASTER_CONFIG_PATH is not None:

    try:
        with open(MASTER_CONFIG_PATH, "r", encoding="utf-8") as file:

            if MASTER_CONFIG_PATH.suffix.lower() == ".json":
                MASTER_CONFIG = json.load(file)

            elif MASTER_CONFIG_PATH.suffix.lower() in {".yaml", ".yml"}:
                MASTER_CONFIG = yaml.safe_load(file) or {}

        if not isinstance(MASTER_CONFIG, dict):
            raise ValueError(
                "Master configuration must contain a dictionary/object."
            )

        MASTER_CONFIG_LOADED = True

        print("Master configuration loaded successfully.")
        print("Configuration path:", MASTER_CONFIG_PATH)

    except Exception as error:
        raise RuntimeError(
            f"Unable to load master configuration: {MASTER_CONFIG_PATH}\n"
            f"Error: {error}"
        ) from error

else:

    print("WARNING: Notebook 00 master configuration was not found.")
    print("Searched project root:", PROJECT_ROOT)
    print("\nConfiguration-like files currently present:")

    discovered_files = []

    for current_root, directories, filenames in os.walk(PROJECT_ROOT):

        directories[:] = [
            directory for directory in directories
            if directory not in {
                ".git",
                "__pycache__",
                ".ipynb_checkpoints",
                "node_modules",
            }
        ]

        for filename in filenames:
            if filename.lower().endswith((".json", ".yaml", ".yml")):
                discovered_files.append(
                    Path(current_root) / filename
                )

    if discovered_files:
        for path in discovered_files[:50]:
            print(" -", path)
    else:
        print("No JSON/YAML configuration files found.")

    print(
        "\nNotebook 17 setup will continue with its explicitly defined "
        "ablation settings. Master configuration loading remains UNVERIFIED."
    )

# -----------------------------------------------------------------------------
# 6. NOTEBOOK 17 ABLATION CONFIGURATION
# -----------------------------------------------------------------------------

ABLATION_CONFIG = {
    "notebook": "Notebook 17 — Ablation Study",
    "framework": "SPP-GAN",

    "master_config_loaded": MASTER_CONFIG_LOADED,
    "master_config_path": (
        str(MASTER_CONFIG_PATH)
        if MASTER_CONFIG_PATH is not None
        else None
    ),

    "master_seed": MASTER_SEED,
    "repetitions": REPETITIONS,
    "repetition_seeds": REPETITION_SEEDS,

    "datasets": DATASETS,
    "variants": VARIANTS,
    "expected_records": EXPECTED_RECORDS,

    "significance_level": 0.05,

    "privacy": {
        "target_epsilon": 5.0,
        "delta_rule": "min(1e-5, 1/N_train)",
        "accountant": "RDP",
        "mechanism": "Gaussian",
        "private_component": "Critic",
        "gradient_clipping": True,
        "batch_size": 128,
        "epochs": 300,
    },

    "execution": {
        "retrain_models": False,
        "overwrite_upstream_artifacts": False,
        "preserve_repetitions": True,
        "validate_artifacts": True,
        "save_intermediate_results": True,
    },
}

# -----------------------------------------------------------------------------
# 7. UPSTREAM ARTIFACT PATHS
# -----------------------------------------------------------------------------

UPSTREAM_PATHS = {
    "fidelity": (
        RESULTS_DIR / "notebooks" / "notebook_14"
    ),

    "utility_candidates": [
        RESULTS_DIR / "notebooks" / "notebook_15",
        RESULTS_DIR / "notebooks" / "notebook_14",
    ],

    "privacy_candidates": [
        RESULTS_DIR / "notebooks" / "notebook_16",
        RESULTS_DIR / "notebooks" / "notebook_15",
    ],
}

# -----------------------------------------------------------------------------
# 8. SAVE CONFIGURATION SNAPSHOT
# -----------------------------------------------------------------------------

CONFIG_SNAPSHOT_PATH = (
    NB17_DIRS["metadata"] /
    "notebook_17_configuration_snapshot.json"
)

CONFIG_SNAPSHOT = {
    "created_at": datetime.now().isoformat(),
    "project_root": str(PROJECT_ROOT),
    "master_config_loaded": MASTER_CONFIG_LOADED,
    "master_config_path": (
        str(MASTER_CONFIG_PATH)
        if MASTER_CONFIG_PATH is not None
        else None
    ),
    "master_config": MASTER_CONFIG,
    "ablation_config": ABLATION_CONFIG,
    "upstream_paths": {
        key: (
            [str(path) for path in value]
            if isinstance(value, list)
            else str(value)
        )
        for key, value in UPSTREAM_PATHS.items()
    },
}

with open(CONFIG_SNAPSHOT_PATH, "w", encoding="utf-8") as file:
    json.dump(CONFIG_SNAPSHOT, file, indent=4, default=str)

# -----------------------------------------------------------------------------
# 9. SAVE DIRECTORY MANIFEST
# -----------------------------------------------------------------------------

MANIFEST_PATH = (
    NB17_DIRS["manifests"] /
    "notebook_17_directory_manifest.csv"
)

with open(
    MANIFEST_PATH,
    "w",
    newline="",
    encoding="utf-8"
) as file:

    writer = csv.writer(file)
    writer.writerow(["directory_name", "directory_path", "exists"])

    for name, directory in NB17_DIRS.items():
        writer.writerow([
            name,
            str(directory),
            directory.exists(),
        ])

# -----------------------------------------------------------------------------
# 10. VALIDATION
# -----------------------------------------------------------------------------

assert PROJECT_ROOT.exists()
assert NB17_ROOT.exists()
assert all(path.exists() for path in NB17_DIRS.values())

assert len(DATASETS) == 3
assert len(REPETITIONS) == 5
assert len(VARIANTS) == 4
assert EXPECTED_RECORDS == 60

assert CONFIG_SNAPSHOT_PATH.exists()
assert MANIFEST_PATH.exists()

# -----------------------------------------------------------------------------
# 11. EXECUTION SUMMARY
# -----------------------------------------------------------------------------

print("\n" + "=" * 75)
print("NOTEBOOK 17 — SECTION 2 VALIDATION")
print("=" * 75)

print("Project root:", PROJECT_ROOT)
print("Notebook root:", NB17_ROOT)
print("Expected records:", EXPECTED_RECORDS)

print("\nMaster configuration loaded:", MASTER_CONFIG_LOADED)
print("Master configuration path:", MASTER_CONFIG_PATH)

print("\nNotebook 17 directories:")
for name, directory in NB17_DIRS.items():
    print(f"{name:15s}: {directory}")

print("\nPrivacy configuration:")
print(json.dumps(ABLATION_CONFIG["privacy"], indent=4))

print("\nExecution controls:")
print(json.dumps(ABLATION_CONFIG["execution"], indent=4))

print("\nConfiguration snapshot:", CONFIG_SNAPSHOT_PATH)
print("Directory manifest:", MANIFEST_PATH)

print("\nDirectory and configuration snapshot validation: PASS")

if MASTER_CONFIG_LOADED:
    print("Section 2 status: PASS")
else:
    print("Section 2 status: WARNING — MASTER CONFIGURATION NOT FOUND")

print("=" * 75)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Master configuration loaded successfully.
Configuration path: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/config/sppgan_model_configuration.json

NOTEBOOK 17 — SECTION 2 VALIDATION
Project root: /content/drive/MyDrive/SPP_GAN_Research
Notebook root: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_17
Expected records: 60

Master configuration loaded: True
Master configuration path: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/config/sppgan_model_configuration.json

Notebook 17 directories:
results        : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_17/results
tables         : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_17/tables
metadata       : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_17/metadata
validation     : /cont

In [25]:

# =============================================================================
# SECTION 3 — DEFINE ABLATION VARIANTS
# Notebook 17 — Ablation Study
# =============================================================================

import pandas as pd

# -----------------------------------------------------------------------------
# 1. DEFINE THE FOUR ABLATION VARIANTS
# -----------------------------------------------------------------------------

VARIANT_REGISTRY = {
    "gan_only": {
        "variant_label": "GAN-only",
        "statistical_guidance": False,
        "marginal_guidance": False,
        "moment_guidance": False,
        "dependency_guidance": False,
        "distribution_guidance": False,
        "categorical_guidance": False,
        "differential_privacy": False,
        "description": (
            "Adversarial GAN without statistical guidance "
            "or differentially private critic optimization."
        ),
    },

    "statistical_guidance_gan": {
        "variant_label": "Statistical Guidance + GAN",
        "statistical_guidance": True,
        "marginal_guidance": True,
        "moment_guidance": True,
        "dependency_guidance": True,
        "distribution_guidance": True,
        "categorical_guidance": True,
        "differential_privacy": False,
        "description": (
            "GAN incorporating marginal, moment, dependency, "
            "and categorical statistical guidance without DP."
        ),
    },

    "gan_dp": {
        "variant_label": "GAN + Differential Privacy",
        "statistical_guidance": False,
        "marginal_guidance": False,
        "moment_guidance": False,
        "dependency_guidance": False,
        "distribution_guidance": False,
        "categorical_guidance": False,
        "differential_privacy": True,
        "description": (
            "GAN using differentially private critic optimization "
            "without statistical guidance."
        ),
    },

    "full_sppgan": {
        "variant_label": "Complete SPP-GAN",
        "statistical_guidance": True,
        "marginal_guidance": True,
        "moment_guidance": True,
        "dependency_guidance": True,
        "distribution_guidance": True,
        "categorical_guidance": True,
        "differential_privacy": True,
        "description": (
            "Complete SPP-GAN integrating statistical guidance "
            "and differentially private critic optimization."
        ),
    },
}

# -----------------------------------------------------------------------------
# 2. VALIDATE VARIANT DEFINITIONS
# -----------------------------------------------------------------------------

EXPECTED_VARIANTS = set(VARIANTS)
REGISTERED_VARIANTS = set(VARIANT_REGISTRY)

assert REGISTERED_VARIANTS == EXPECTED_VARIANTS, (
    "Variant registry does not match Section 1 VARIANTS.\n"
    f"Missing: {EXPECTED_VARIANTS - REGISTERED_VARIANTS}\n"
    f"Unexpected: {REGISTERED_VARIANTS - EXPECTED_VARIANTS}"
)

GUIDANCE_FLAGS = [
    "marginal_guidance",
    "moment_guidance",
    "dependency_guidance",
    "categorical_guidance",
]

for variant_name, config in VARIANT_REGISTRY.items():

    expected_statistical_guidance = any(
        config[flag] for flag in GUIDANCE_FLAGS
    )

    assert config["statistical_guidance"] == (
        expected_statistical_guidance
    ), (
        f"Inconsistent statistical guidance flags: {variant_name}"
    )

    assert config["distribution_guidance"] == (
        config["statistical_guidance"]
    ), (
        f"Inconsistent distribution guidance flag: {variant_name}"
    )

    assert isinstance(config["differential_privacy"], bool)

# -----------------------------------------------------------------------------
# 3. CREATE VARIANT REGISTRY DATAFRAME
# -----------------------------------------------------------------------------

variant_df = pd.DataFrame([
    {
        "variant": variant_name,
        **config,
    }
    for variant_name, config in VARIANT_REGISTRY.items()
])

assert len(variant_df) == len(VARIANTS) == 4
assert variant_df["variant"].is_unique
assert not variant_df.isnull().any().any()

# -----------------------------------------------------------------------------
# 4. SAVE REGISTRY
# -----------------------------------------------------------------------------

VARIANT_REGISTRY_PATH = REGISTRY_PATH

variant_df.to_csv(
    VARIANT_REGISTRY_PATH,
    index=False,
)

# Save a metadata copy for traceability.
VARIANT_METADATA_PATH = (
    NB17_DIRS["metadata"] /
    "ablation_variant_definitions.csv"
)

variant_df.to_csv(
    VARIANT_METADATA_PATH,
    index=False,
)

# -----------------------------------------------------------------------------
# 5. VERIFY SAVED REGISTRY
# -----------------------------------------------------------------------------

saved_variant_df = pd.read_csv(VARIANT_REGISTRY_PATH)

assert len(saved_variant_df) == 4
assert saved_variant_df["variant"].is_unique
assert set(saved_variant_df["variant"]) == EXPECTED_VARIANTS

assert VARIANT_REGISTRY_PATH.exists()
assert VARIANT_METADATA_PATH.exists()

# -----------------------------------------------------------------------------
# 6. DISPLAY VALIDATED VARIANT SUMMARY
# -----------------------------------------------------------------------------

DISPLAY_COLUMNS = [
    "variant",
    "variant_label",
    "statistical_guidance",
    "marginal_guidance",
    "moment_guidance",
    "dependency_guidance",
    "distribution_guidance",
    "categorical_guidance",
    "differential_privacy",
]

print("\n" + "=" * 100)
print("NOTEBOOK 17 — ABLATION VARIANT REGISTRY")
print("=" * 100)

print(
    saved_variant_df[DISPLAY_COLUMNS].to_string(index=False)
)

print("\nRegistered variants:", len(saved_variant_df))
print("Expected variants:", len(VARIANTS))
print("Variant names match Section 1: PASS")
print("Statistical guidance consistency: PASS")
print("Saved registry verification: PASS")

print("\nRegistry:", VARIANT_REGISTRY_PATH)
print("Metadata copy:", VARIANT_METADATA_PATH)

print("\nSECTION 3 VALIDATION: PASS")
print("=" * 100)



NOTEBOOK 17 — ABLATION VARIANT REGISTRY
                 variant              variant_label  statistical_guidance  marginal_guidance  moment_guidance  dependency_guidance  distribution_guidance  categorical_guidance  differential_privacy
                gan_only                   GAN-only                 False              False            False                False                  False                 False                 False
statistical_guidance_gan Statistical Guidance + GAN                  True               True             True                 True                   True                  True                 False
                  gan_dp GAN + Differential Privacy                 False              False            False                False                  False                 False                  True
             full_sppgan           Complete SPP-GAN                  True               True             True                 True                   True              

In [27]:

# =============================================================================
# SECTION 4 — VALIDATE ABLATION MODELS AND REGISTRY
# Notebook 17 — Ablation Study
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

# -----------------------------------------------------------------------------
# 1. DEFINE SEPARATE REGISTRY PATHS
# -----------------------------------------------------------------------------

# Section 3 registry: definitions of the four ablation variants.
VARIANT_REGISTRY_PATH = (
    NB17_DIRS["registries"] /
    "ablation_variant_registry.csv"
)

# Section 4 registry: actual experiment outputs.
EXPERIMENT_REGISTRY_PATH = (
    NB17_DIRS["registries"] /
    "ablation_experiment_registry.csv"
)

VALIDATION_PATH = (
    NB17_DIRS["validation"] /
    "registry_validation.csv"
)

REQUIRED_REGISTRY_COLUMNS = [
    "dataset",
    "repetition_id",
    "variant",
    "synthetic_path",
]

OPTIONAL_REGISTRY_COLUMNS = [
    "model_path",
    "model_sha256",
    "achieved_epsilon",
    "delta",
    "privacy_scope",
    "training_runtime_seconds",
    "generation_runtime_seconds",
    "training_seed",
    "generation_seed",
    "real_test_path",
]

# -----------------------------------------------------------------------------
# 2. VALIDATE THE VARIANT-DEFINITION REGISTRY
# -----------------------------------------------------------------------------

if not VARIANT_REGISTRY_PATH.exists():
    raise FileNotFoundError(
        "Section 3 variant registry is missing:\n"
        f"{VARIANT_REGISTRY_PATH}"
    )

variant_registry_check = pd.read_csv(VARIANT_REGISTRY_PATH)

assert "variant" in variant_registry_check.columns, (
    "The variant-definition registry must contain a 'variant' column."
)

assert set(variant_registry_check["variant"].astype(str)) == set(VARIANTS), (
    "Section 3 variant definitions do not match Section 1 VARIANTS."
)

print("Variant-definition registry: PASS")

# -----------------------------------------------------------------------------
# 3. DISCOVER A COMPATIBLE EXPERIMENT REGISTRY
# -----------------------------------------------------------------------------

def has_required_columns(file_path):
    """Check whether a CSV has the required experiment-registry schema."""

    try:
        columns = pd.read_csv(file_path, nrows=0).columns.tolist()
        return all(
            column in columns
            for column in REQUIRED_REGISTRY_COLUMNS
        )
    except Exception:
        return False


if EXPERIMENT_REGISTRY_PATH.exists():

    print(
        "Using existing Notebook 17 experiment registry:",
        EXPERIMENT_REGISTRY_PATH
    )

else:

    search_roots = [
        RESULTS_DIR / "notebooks" / "notebook_13",
        RESULTS_DIR / "notebooks" / "notebook_17",
    ]

    discovered_registries = []

    for search_root in search_roots:

        if not search_root.exists():
            continue

        for candidate in search_root.rglob("*.csv"):

            # Exclude the Section 3 variant-definition registry.
            if candidate.resolve() == VARIANT_REGISTRY_PATH.resolve():
                continue

            if candidate.resolve() == VALIDATION_PATH.resolve():
                continue

            if has_required_columns(candidate):
                discovered_registries.append(candidate)

    if discovered_registries:

        # Prefer an existing experiment registry in Notebook 17,
        # then a compatible registry from Notebook 13.
        discovered_registries.sort(
            key=lambda path: (
                0 if "notebook_17" in str(path) else 1,
                len(str(path)),
            )
        )

        source_registry = discovered_registries[0]

        registry_df = pd.read_csv(source_registry)

        print("Compatible experiment registry discovered.")
        print("Source:", source_registry)

        # Copy into Notebook 17 without modifying the source.
        registry_df.to_csv(
            EXPERIMENT_REGISTRY_PATH,
            index=False,
        )

    else:

        print("No compatible experiment registry was found.")
        print("Creating a separate experiment-registry template.")

        template_records = []

        for dataset in DATASETS:
            for repetition in REPETITIONS:
                for variant in VARIANTS:

                    template_records.append({
                        "dataset": dataset,
                        "repetition_id": repetition,
                        "variant": variant,
                        "synthetic_path": "",
                        "real_test_path": "",
                    })

        registry_df = pd.DataFrame(template_records)

        registry_df.to_csv(
            EXPERIMENT_REGISTRY_PATH,
            index=False,
        )

        print(
            "Template created. Populate synthetic_path with the "
            "actual generated dataset paths before downstream evaluation."
        )

# -----------------------------------------------------------------------------
# 4. LOAD AND VALIDATE EXPERIMENT REGISTRY
# -----------------------------------------------------------------------------

registry_df = pd.read_csv(EXPERIMENT_REGISTRY_PATH)

missing_columns = [
    column
    for column in REQUIRED_REGISTRY_COLUMNS
    if column not in registry_df.columns
]

if missing_columns:
    raise ValueError(
        "Experiment registry has an incompatible schema.\n"
        f"Missing columns: {missing_columns}\n"
        f"Registry: {EXPERIMENT_REGISTRY_PATH}"
    )

for column in OPTIONAL_REGISTRY_COLUMNS:
    if column not in registry_df.columns:
        registry_df[column] = np.nan

# -----------------------------------------------------------------------------
# 5. NORMALIZE REGISTRY FIELDS
# -----------------------------------------------------------------------------

registry_df["dataset"] = (
    registry_df["dataset"]
    .fillna("")
    .astype(str)
    .str.strip()
)

registry_df["variant"] = (
    registry_df["variant"]
    .fillna("")
    .astype(str)
    .str.strip()
)

registry_df["repetition_id"] = pd.to_numeric(
    registry_df["repetition_id"],
    errors="coerce",
).astype("Int64")

registry_df["synthetic_path"] = (
    registry_df["synthetic_path"]
    .fillna("")
    .astype(str)
    .str.strip()
)

registry_df["real_test_path"] = (
    registry_df["real_test_path"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# -----------------------------------------------------------------------------
# 6. CHECK DUPLICATE EXPERIMENT COMBINATIONS
# -----------------------------------------------------------------------------

duplicate_mask = registry_df.duplicated(
    subset=["dataset", "repetition_id", "variant"],
    keep=False,
)

duplicate_count = int(duplicate_mask.sum())

# -----------------------------------------------------------------------------
# 7. CHECK EXPECTED EXPERIMENT COVERAGE
# -----------------------------------------------------------------------------

expected_keys = {
    (dataset, repetition, variant)
    for dataset in DATASETS
    for repetition in REPETITIONS
    for variant in VARIANTS
}

actual_keys = {
    (dataset, repetition, variant)
    for dataset, repetition, variant in zip(
        registry_df["dataset"],
        registry_df["repetition_id"],
        registry_df["variant"],
    )
    if pd.notna(repetition)
}

missing_keys = expected_keys - actual_keys
unexpected_keys = actual_keys - expected_keys

# -----------------------------------------------------------------------------
# 8. VALIDATE SYNTHETIC OUTPUT FILES
# -----------------------------------------------------------------------------

def validate_file(path_value):
    """Return whether a non-empty path points to an existing file."""

    if not isinstance(path_value, str) or not path_value.strip():
        return False

    try:
        return Path(path_value).is_file()
    except (OSError, ValueError, TypeError):
        return False


registry_df["synthetic_exists"] = (
    registry_df["synthetic_path"].apply(validate_file)
)

registry_df["synthetic_size_bytes"] = (
    registry_df["synthetic_path"].apply(
        lambda path: (
            Path(path).stat().st_size
            if validate_file(path)
            else 0
        )
    )
)

registry_df["registry_status"] = np.where(
    registry_df["synthetic_exists"],
    "PASS",
    "MISSING_SYNTHETIC",
)

# -----------------------------------------------------------------------------
# 9. VALIDATE VARIANT AND DATASET NAMES
# -----------------------------------------------------------------------------

registry_df["valid_dataset"] = (
    registry_df["dataset"].isin(DATASETS)
)

registry_df["valid_variant"] = (
    registry_df["variant"].isin(VARIANTS)
)

registry_df["valid_repetition"] = (
    registry_df["repetition_id"].isin(REPETITIONS)
)

registry_df["combination_valid"] = (
    registry_df["valid_dataset"]
    & registry_df["valid_variant"]
    & registry_df["valid_repetition"]
)

# -----------------------------------------------------------------------------
# 10. SAVE VALIDATION RESULTS
# -----------------------------------------------------------------------------

registry_df.to_csv(
    VALIDATION_PATH,
    index=False,
)

# -----------------------------------------------------------------------------
# 11. VALIDATION SUMMARY
# -----------------------------------------------------------------------------

existing_synthetic_count = int(
    registry_df["synthetic_exists"].sum()
)

missing_synthetic_count = (
    len(registry_df) - existing_synthetic_count
)

coverage_complete = (
    len(registry_df) == EXPECTED_RECORDS
    and not missing_keys
    and not unexpected_keys
    and duplicate_count == 0
)

files_complete = (
    len(registry_df) == EXPECTED_RECORDS
    and registry_df["synthetic_exists"].all()
)

print("\n" + "=" * 100)
print("NOTEBOOK 17 — SECTION 4 REGISTRY VALIDATION")
print("=" * 100)

print("Variant registry:", VARIANT_REGISTRY_PATH)
print("Experiment registry:", EXPERIMENT_REGISTRY_PATH)
print("Validation output:", VALIDATION_PATH)

print("\nRegistry rows:", len(registry_df))
print("Expected combinations:", EXPECTED_RECORDS)
print("Missing combinations:", len(missing_keys))
print("Unexpected combinations:", len(unexpected_keys))
print("Duplicate records:", duplicate_count)

print("\nExisting synthetic outputs:", existing_synthetic_count)
print("Missing synthetic outputs:", missing_synthetic_count)

print("\nCoverage validation:", "PASS" if coverage_complete else "INCOMPLETE")
print("Synthetic file validation:", "PASS" if files_complete else "INCOMPLETE")

if coverage_complete and files_complete:
    print("\nSECTION 4 STATUS: PASS")

else:
    print("\nSECTION 4 STATUS: INCOMPLETE")
    print(
        "Populate or correct the experiment registry using actual "
        "Notebook 13 outputs before proceeding to evaluation."
    )

print("=" * 100)


Variant-definition registry: PASS
No compatible experiment registry was found.
Creating a separate experiment-registry template.
Template created. Populate synthetic_path with the actual generated dataset paths before downstream evaluation.

NOTEBOOK 17 — SECTION 4 REGISTRY VALIDATION
Variant registry: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_17/registries/ablation_variant_registry.csv
Experiment registry: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_17/registries/ablation_experiment_registry.csv
Validation output: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_17/validation/registry_validation.csv

Registry rows: 60
Expected combinations: 60
Missing combinations: 0
Unexpected combinations: 0
Duplicate records: 0

Existing synthetic outputs: 0
Missing synthetic outputs: 60

Coverage validation: PASS
Synthetic file validation: INCOMPLETE

SECTION 4 STATUS: INCOMPLETE
Populate or correct the experiment registry using act

In [29]:

# =============================================================================
# SECTION 5 — LOAD SYNTHETIC OUTPUTS
# RAM-COMPATIBLE: LOAD ONE DATASET/VARIANT AT A TIME
# =============================================================================

import gc
import pandas as pd
import numpy as np
from pathlib import Path

# -----------------------------------------------------------------------------
# 1. DEFINE SAFE CSV LOADER
# -----------------------------------------------------------------------------

def load_csv_safely(path, columns=None):
    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(
            f"Synthetic or real dataset not found: {path}"
        )

    if path.suffix.lower() != ".csv":
        raise ValueError(
            f"Expected CSV artifact, received: {path}"
        )

    return pd.read_csv(
        path,
        usecols=columns,
        low_memory=True,
    )


# -----------------------------------------------------------------------------
# 2. RESOLVE REAL TEST DATA PATH
# -----------------------------------------------------------------------------

def resolve_real_test_path(row):

    candidate = row.get("real_test_path", None)

    if pd.notna(candidate):
        candidate = str(candidate).strip()

        if candidate and Path(candidate).is_file():
            return Path(candidate)

    dataset = str(row["dataset"]).strip()

    configured = REAL_TEST_PATHS.get(dataset)

    if configured:
        configured = Path(configured)

        if configured.is_file():
            return configured

    raise FileNotFoundError(
        f"Real test data path missing for {dataset}. "
        "Set real_test_path in the experiment registry or "
        "REAL_TEST_PATHS in Section 2."
    )


# -----------------------------------------------------------------------------
# 3. VALIDATE EXPERIMENT REGISTRY AVAILABILITY
# -----------------------------------------------------------------------------

if "registry_df" not in globals():
    raise RuntimeError(
        "Experiment registry is unavailable. Run Section 4 first."
    )

required_columns = [
    "dataset",
    "repetition_id",
    "variant",
    "synthetic_path",
]

missing_columns = [
    column for column in required_columns
    if column not in registry_df.columns
]

if missing_columns:
    raise ValueError(
        f"Experiment registry is missing columns: {missing_columns}"
    )

# Recalculate file availability to avoid relying on stale Section 4 results.
registry_df["synthetic_path"] = (
    registry_df["synthetic_path"]
    .fillna("")
    .astype(str)
    .str.strip()
)

registry_df["synthetic_exists"] = (
    registry_df["synthetic_path"].apply(
        lambda path: bool(path) and Path(path).is_file()
    )
)

registry_df["synthetic_size_bytes"] = (
    registry_df["synthetic_path"].apply(
        lambda path: (
            Path(path).stat().st_size
            if path and Path(path).is_file()
            else 0
        )
    )
)

# -----------------------------------------------------------------------------
# 4. BUILD SYNTHETIC OUTPUT MANIFEST
# -----------------------------------------------------------------------------

MANIFEST_COLUMNS = [
    "dataset",
    "repetition_id",
    "variant",
    "synthetic_path",
    "synthetic_size_bytes",
    "real_test_path",
]

synthetic_manifest = []

available_rows = registry_df.loc[
    registry_df["synthetic_exists"]
].copy()

for _, row in available_rows.iterrows():

    synthetic_manifest.append({
        "dataset": str(row["dataset"]),
        "repetition_id": int(row["repetition_id"]),
        "variant": str(row["variant"]),
        "synthetic_path": str(row["synthetic_path"]),
        "synthetic_size_bytes": int(
            row["synthetic_size_bytes"]
        ),
        "real_test_path": (
            str(row["real_test_path"])
            if pd.notna(row.get("real_test_path"))
            else ""
        ),
    })

synthetic_manifest_df = pd.DataFrame(
    synthetic_manifest,
    columns=MANIFEST_COLUMNS,
)

# -----------------------------------------------------------------------------
# 5. SAVE MANIFEST
# -----------------------------------------------------------------------------

SYNTHETIC_MANIFEST_PATH = (
    NB17_DIRS["manifests"] /
    "synthetic_ablation_manifest.csv"
)

synthetic_manifest_df.to_csv(
    SYNTHETIC_MANIFEST_PATH,
    index=False,
)

# -----------------------------------------------------------------------------
# 6. SUMMARIZE AVAILABLE OUTPUTS
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("NOTEBOOK 17 — SECTION 5 SYNTHETIC OUTPUT MANIFEST")
print("=" * 100)

print("Registered experiment combinations:", len(registry_df))
print("Available synthetic outputs:", len(synthetic_manifest_df))
print(
    "Missing synthetic outputs:",
    int((~registry_df["synthetic_exists"]).sum())
)

if not synthetic_manifest_df.empty:

    print("\nAvailable outputs by dataset and variant:")

    print(
        synthetic_manifest_df.groupby(
            ["dataset", "variant"]
        ).size().to_string()
    )

else:

    print("\nNo synthetic output files are currently available.")
    print(
        "The manifest has been saved with its expected columns "
        "and zero records."
    )

# -----------------------------------------------------------------------------
# 7. VALIDATE MANIFEST
# -----------------------------------------------------------------------------

assert list(synthetic_manifest_df.columns) == MANIFEST_COLUMNS
assert SYNTHETIC_MANIFEST_PATH.exists()

assert len(synthetic_manifest_df) == len(available_rows)

if not synthetic_manifest_df.empty:

    assert synthetic_manifest_df["synthetic_path"].apply(
        lambda path: Path(path).is_file()
    ).all()

    assert synthetic_manifest_df["synthetic_size_bytes"].gt(0).all()

# -----------------------------------------------------------------------------
# 8. DOWNSTREAM READINESS
# -----------------------------------------------------------------------------

SYNTHETIC_OUTPUTS_READY = (
    len(synthetic_manifest_df) == EXPECTED_RECORDS
    and synthetic_manifest_df["synthetic_path"].apply(
        lambda path: Path(path).is_file()
    ).all()
)

print("\nManifest:", SYNTHETIC_MANIFEST_PATH)
print("Manifest validation: PASS")
print(
    "Synthetic outputs ready for downstream evaluation:",
    SYNTHETIC_OUTPUTS_READY
)

if not SYNTHETIC_OUTPUTS_READY:
    print(
        "\nSECTION 5 STATUS: INCOMPLETE — actual synthetic outputs "
        "must be registered before evaluation."
    )
else:
    print("\nSECTION 5 STATUS: PASS")

print("=" * 100)

# -----------------------------------------------------------------------------
# 9. RELEASE TEMPORARY MEMORY
# -----------------------------------------------------------------------------

del synthetic_manifest
gc.collect()



NOTEBOOK 17 — SECTION 5 SYNTHETIC OUTPUT MANIFEST
Registered experiment combinations: 60
Available synthetic outputs: 0
Missing synthetic outputs: 60

No synthetic output files are currently available.
The manifest has been saved with its expected columns and zero records.

Manifest: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_17/manifests/synthetic_ablation_manifest.csv
Manifest validation: PASS
Synthetic outputs ready for downstream evaluation: False

SECTION 5 STATUS: INCOMPLETE — actual synthetic outputs must be registered before evaluation.


145

In [31]:

# =============================================================================
# SECTION 6 — STATISTICAL-GUIDANCE ABLATION
# RAM-COMPATIBLE: ONE DATASET/REPETITION/VARIANT AT A TIME
# =============================================================================

import gc
import numpy as np
import pandas as pd

from pathlib import Path
from scipy.stats import ks_2samp, wasserstein_distance
from scipy.spatial.distance import jensenshannon

EPSILON = 1e-10

FIDELITY_OUTPUT_PATH = (
    NB17_DIRS["results"] /
    "ablation_fidelity_component_results.csv"
)

FIDELITY_COLUMNS = [
    "dataset",
    "repetition_id",
    "variant",
    "variant_label",
    "real_test_path",
    "synthetic_path",
    "sample_size",
    "numeric_features_evaluated",
    "categorical_features_evaluated",
    "mean_ks_statistic",
    "mean_wasserstein_distance",
    "mean_moment_mean_error",
    "mean_moment_std_error",
    "mean_js_distance",
    "mean_total_variation_distance",
    "correlation_frobenius_error",
    "mean_absolute_correlation_error",
]

# -----------------------------------------------------------------------------
# 1. NUMERIC DATA VALIDATION
# -----------------------------------------------------------------------------

def safe_numeric(series):

    values = pd.to_numeric(series, errors="coerce")

    return (
        values.replace([np.inf, -np.inf], np.nan)
        .dropna()
        .to_numpy(dtype=np.float64)
    )


# -----------------------------------------------------------------------------
# 2. NUMERIC DISTRIBUTION FIDELITY
# -----------------------------------------------------------------------------

def calculate_numeric_fidelity(real, synthetic):

    real = safe_numeric(real)
    synthetic = safe_numeric(synthetic)

    if len(real) == 0 or len(synthetic) == 0:
        return {
            "ks": np.nan,
            "wasserstein": np.nan,
            "moment_mean_error": np.nan,
            "moment_std_error": np.nan,
        }

    real_mean = np.mean(real)
    synthetic_mean = np.mean(synthetic)

    real_std = np.std(real)
    synthetic_std = np.std(synthetic)

    mean_error = (
        abs(real_mean - synthetic_mean) /
        (abs(real_mean) + EPSILON)
    )

    std_error = (
        abs(real_std - synthetic_std) /
        (real_std + EPSILON)
    )

    return {
        "ks": float(ks_2samp(real, synthetic).statistic),
        "wasserstein": float(
            wasserstein_distance(real, synthetic)
        ),
        "moment_mean_error": float(mean_error),
        "moment_std_error": float(std_error),
    }


# -----------------------------------------------------------------------------
# 3. CATEGORICAL DISTRIBUTION FIDELITY
# -----------------------------------------------------------------------------

def calculate_categorical_fidelity(real, synthetic):

    real = (
        real.astype("string")
        .fillna("__MISSING__")
    )

    synthetic = (
        synthetic.astype("string")
        .fillna("__MISSING__")
    )

    categories = sorted(
        set(real.unique()).union(set(synthetic.unique()))
    )

    if not categories:
        return {
            "js_distance": np.nan,
            "total_variation": np.nan,
        }

    p = (
        real.value_counts(normalize=True)
        .reindex(categories, fill_value=0)
        .to_numpy(dtype=float)
    )

    q = (
        synthetic.value_counts(normalize=True)
        .reindex(categories, fill_value=0)
        .to_numpy(dtype=float)
    )

    js = jensenshannon(p, q, base=2)
    tv = 0.5 * np.abs(p - q).sum()

    return {
        "js_distance": float(js),
        "total_variation": float(tv),
    }


# -----------------------------------------------------------------------------
# 4. CORRELATION STRUCTURE FIDELITY
# -----------------------------------------------------------------------------

def calculate_correlation_error(real, synthetic):

    common = [
        column for column in real.columns
        if column in synthetic.columns
    ]

    numeric = [
        column for column in common
        if pd.api.types.is_numeric_dtype(real[column])
        and pd.api.types.is_numeric_dtype(synthetic[column])
    ]

    if len(numeric) < 2:
        return np.nan, np.nan

    real_corr = real[numeric].corr(method="pearson")
    synthetic_corr = synthetic[numeric].corr(method="pearson")

    real_corr = real_corr.fillna(0).to_numpy(dtype=float)
    synthetic_corr = synthetic_corr.fillna(0).to_numpy(dtype=float)

    difference = real_corr - synthetic_corr

    frobenius = (
        np.linalg.norm(difference, ord="fro") /
        (len(numeric) ** 2)
    )

    mae = np.mean(np.abs(difference))

    return float(frobenius), float(mae)


# -----------------------------------------------------------------------------
# 5. AGGREGATE DISTRIBUTION FIDELITY
# -----------------------------------------------------------------------------

def evaluate_distribution_pair(real, synthetic, metadata):

    common = [
        column for column in real.columns
        if column in synthetic.columns
    ]

    numeric_cols = [
        column for column in common
        if pd.api.types.is_numeric_dtype(real[column])
        and pd.api.types.is_numeric_dtype(synthetic[column])
    ]

    categorical_cols = [
        column for column in common
        if column not in numeric_cols
    ]

    numeric_results = [
        calculate_numeric_fidelity(
            real[column],
            synthetic[column],
        )
        for column in numeric_cols
    ]

    categorical_results = [
        calculate_categorical_fidelity(
            real[column],
            synthetic[column],
        )
        for column in categorical_cols
    ]

    def average_metric(records, key):

        values = [
            record[key]
            for record in records
            if np.isfinite(record[key])
        ]

        return float(np.mean(values)) if values else np.nan

    corr_fro, corr_mae = calculate_correlation_error(
        real[numeric_cols],
        synthetic[numeric_cols],
    ) if len(numeric_cols) >= 2 else (np.nan, np.nan)

    return {
        **metadata,
        "sample_size": len(synthetic),
        "numeric_features_evaluated": len(numeric_cols),
        "categorical_features_evaluated": len(categorical_cols),
        "mean_ks_statistic": average_metric(
            numeric_results, "ks"
        ),
        "mean_wasserstein_distance": average_metric(
            numeric_results, "wasserstein"
        ),
        "mean_moment_mean_error": average_metric(
            numeric_results, "moment_mean_error"
        ),
        "mean_moment_std_error": average_metric(
            numeric_results, "moment_std_error"
        ),
        "mean_js_distance": average_metric(
            categorical_results, "js_distance"
        ),
        "mean_total_variation_distance": average_metric(
            categorical_results, "total_variation"
        ),
        "correlation_frobenius_error": corr_fro,
        "mean_absolute_correlation_error": corr_mae,
    }


# -----------------------------------------------------------------------------
# 6. PRE-EVALUATION VALIDATION
# -----------------------------------------------------------------------------

if "registry_df" not in globals():
    raise RuntimeError("Run Section 4 before Section 6.")

if "load_csv_safely" not in globals():
    raise RuntimeError("Run Section 5 before Section 6.")

if "resolve_real_test_path" not in globals():
    raise RuntimeError("Real test path resolver is unavailable.")

required_columns = [
    "dataset",
    "repetition_id",
    "variant",
    "synthetic_path",
]

missing_columns = [
    column for column in required_columns
    if column not in registry_df.columns
]

if missing_columns:
    raise ValueError(
        f"Experiment registry is missing columns: {missing_columns}"
    )

# Recheck file availability immediately before evaluation.
registry_df["synthetic_exists"] = (
    registry_df["synthetic_path"]
    .fillna("")
    .astype(str)
    .str.strip()
    .apply(
        lambda path: bool(path) and Path(path).is_file()
    )
)

available_registry = registry_df.loc[
    registry_df["synthetic_exists"]
].copy()

# -----------------------------------------------------------------------------
# 7. EVALUATE AVAILABLE EXPERIMENTS
# -----------------------------------------------------------------------------

fidelity_records = []
fidelity_failures = []

if available_registry.empty:

    print(
        "BLOCKED: No synthetic outputs are available. "
        "No fidelity evaluations can be performed."
    )

else:

    for dataset in DATASETS:

        for repetition in REPETITIONS:

            subset = available_registry[
                (available_registry["dataset"] == dataset)
                & (
                    available_registry["repetition_id"]
                    == repetition
                )
            ]

            for variant in VARIANTS:

                row_match = subset[
                    subset["variant"] == variant
                ]

                if row_match.empty:
                    continue

                row = row_match.iloc[0]

                try:

                    synthetic = load_csv_safely(
                        row["synthetic_path"]
                    )

                    real_path = resolve_real_test_path(row)

                    real = load_csv_safely(real_path)

                    common = [
                        column for column in real.columns
                        if column in synthetic.columns
                    ]

                    if not common:
                        raise ValueError(
                            "No common real/synthetic columns."
                        )

                    target_candidates = {
                        "adult_income": "income",
                        "bank_marketing": "y",
                        "diabetes_130us": "readmitted",
                    }

                    target = target_candidates.get(dataset)

                    feature_columns = [
                        column for column in common
                        if column != target
                    ]

                    if not feature_columns:
                        raise ValueError(
                            "No common feature columns remain "
                            "after excluding the target."
                        )

                    result = evaluate_distribution_pair(
                        real[feature_columns],
                        synthetic[feature_columns],
                        {
                            "dataset": dataset,
                            "repetition_id": int(repetition),
                            "variant": variant,
                            "variant_label": VARIANT_REGISTRY[
                                variant
                            ]["variant_label"],
                            "real_test_path": str(real_path),
                            "synthetic_path": str(
                                row["synthetic_path"]
                            ),
                        },
                    )

                    fidelity_records.append(result)

                    del real, synthetic
                    gc.collect()

                except Exception as exc:

                    fidelity_failures.append({
                        "dataset": dataset,
                        "repetition_id": repetition,
                        "variant": variant,
                        "error": str(exc),
                    })

                    print(
                        "Fidelity evaluation failed:",
                        dataset,
                        repetition,
                        variant,
                        str(exc),
                    )

# -----------------------------------------------------------------------------
# 8. SAVE RESULTS WITH A FIXED SCHEMA
# -----------------------------------------------------------------------------

fidelity_df = pd.DataFrame(
    fidelity_records,
    columns=FIDELITY_COLUMNS,
)

fidelity_df.to_csv(
    FIDELITY_OUTPUT_PATH,
    index=False,
)

FAILURE_OUTPUT_PATH = (
    NB17_DIRS["validation"] /
    "ablation_fidelity_failures.csv"
)

pd.DataFrame(
    fidelity_failures,
    columns=[
        "dataset",
        "repetition_id",
        "variant",
        "error",
    ],
).to_csv(
    FAILURE_OUTPUT_PATH,
    index=False,
)

# -----------------------------------------------------------------------------
# 9. VALIDATION AND STATUS
# -----------------------------------------------------------------------------

successful_evaluations = len(fidelity_df)
failed_evaluations = len(fidelity_failures)

evaluation_status = (
    "PASS"
    if successful_evaluations == EXPECTED_RECORDS
    and failed_evaluations == 0
    else "INCOMPLETE"
    if successful_evaluations > 0
    else "BLOCKED"
)

print("\n" + "=" * 100)
print("NOTEBOOK 17 — SECTION 6 STATISTICAL-GUIDANCE ABLATION")
print("=" * 100)

print("Available synthetic outputs:", len(available_registry))
print("Successful fidelity evaluations:", successful_evaluations)
print("Failed fidelity evaluations:", failed_evaluations)
print("Expected experiment records:", EXPECTED_RECORDS)

print("\nFidelity results:", FIDELITY_OUTPUT_PATH)
print("Failure report:", FAILURE_OUTPUT_PATH)

print("\nSECTION 6 STATUS:", evaluation_status)

if evaluation_status == "BLOCKED":
    print(
        "Register the actual synthetic CSV outputs in the "
        "Section 4 experiment registry before rerunning Section 6."
    )

elif evaluation_status == "INCOMPLETE":
    print(
        "Some evaluations remain incomplete. Resolve missing "
        "outputs or evaluation failures before interpreting results."
    )

else:
    print("All expected fidelity evaluations completed successfully.")

print("=" * 100)

gc.collect()


BLOCKED: No synthetic outputs are available. No fidelity evaluations can be performed.

NOTEBOOK 17 — SECTION 6 STATISTICAL-GUIDANCE ABLATION
Available synthetic outputs: 0
Successful fidelity evaluations: 0
Failed fidelity evaluations: 0
Expected experiment records: 60

Fidelity results: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_17/results/ablation_fidelity_component_results.csv
Failure report: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_17/validation/ablation_fidelity_failures.csv

SECTION 6 STATUS: BLOCKED
Register the actual synthetic CSV outputs in the Section 4 experiment registry before rerunning Section 6.


15

In [33]:

# =============================================================================
# SECTION 7 — DEPENDENCY-GUIDANCE ABLATION
# RAM-COMPATIBLE: USE SECTION 6 FIDELITY RESULTS
# =============================================================================

import gc
import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# 1. DEFINE OUTPUT PATHS AND SCHEMAS
# -----------------------------------------------------------------------------

DEPENDENCY_OUTPUT_PATH = (
    NB17_DIRS["results"] /
    "dependency_guidance_ablation.csv"
)

DEPENDENCY_SUMMARY_PATH = (
    NB17_DIRS["tables"] /
    "dependency_guidance_summary.csv"
)

DEPENDENCY_COLUMNS = [
    "dataset",
    "repetition_id",
    "variant",
    "dependency_error",
    "mean_absolute_correlation_error",
    "dependency_guidance_enabled",
]

SUMMARY_COLUMNS = [
    "dataset",
    "variant",
    "variant_label",
    "dependency_guidance_enabled",
    "mean_dependency_error",
    "std_dependency_error",
    "mean_correlation_error",
    "valid_runs",
]

# -----------------------------------------------------------------------------
# 2. VALIDATE SECTION 6 INPUT
# -----------------------------------------------------------------------------

if "fidelity_df" not in globals():
    raise RuntimeError(
        "Section 6 fidelity results are unavailable. Run Section 6 first."
    )

if "VARIANT_REGISTRY" not in globals():
    raise RuntimeError(
        "Variant registry is unavailable. Run Section 3 first."
    )

REQUIRED_FIDELITY_COLUMNS = [
    "dataset",
    "repetition_id",
    "variant",
    "correlation_frobenius_error",
    "mean_absolute_correlation_error",
]

missing_columns = [
    column
    for column in REQUIRED_FIDELITY_COLUMNS
    if column not in fidelity_df.columns
]

# Handle an empty DataFrame that was created without a schema.
if missing_columns and fidelity_df.empty:

    print(
        "WARNING: Section 6 contains no fidelity records. "
        "Creating schema-compatible empty dependency outputs."
    )

    fidelity_input = pd.DataFrame(
        columns=REQUIRED_FIDELITY_COLUMNS
    )

elif missing_columns:

    raise ValueError(
        "Section 6 fidelity results have an incompatible schema. "
        f"Missing columns: {missing_columns}"
    )

else:

    fidelity_input = fidelity_df.copy()

# -----------------------------------------------------------------------------
# 3. EXTRACT DEPENDENCY-GUIDANCE METRICS
# -----------------------------------------------------------------------------

dependency_records = []

for _, row in fidelity_input.iterrows():

    variant = str(row["variant"])

    if variant not in VARIANT_REGISTRY:
        raise ValueError(
            f"Unregistered ablation variant: {variant}"
        )

    dependency_records.append({
        "dataset": row["dataset"],
        "repetition_id": row["repetition_id"],
        "variant": variant,
        "dependency_error": row[
            "correlation_frobenius_error"
        ],
        "mean_absolute_correlation_error": row[
            "mean_absolute_correlation_error"
        ],
        "dependency_guidance_enabled": VARIANT_REGISTRY[
            variant
        ]["dependency_guidance"],
    })

dependency_df = pd.DataFrame(
    dependency_records,
    columns=DEPENDENCY_COLUMNS,
)

# -----------------------------------------------------------------------------
# 4. SAVE COMPONENT-LEVEL RESULTS
# -----------------------------------------------------------------------------

dependency_df.to_csv(
    DEPENDENCY_OUTPUT_PATH,
    index=False,
)

# -----------------------------------------------------------------------------
# 5. CREATE COMPLETE DATASET-VARIANT SUMMARY
# -----------------------------------------------------------------------------

summary_records = []

for dataset in DATASETS:

    for variant in VARIANTS:

        subset = dependency_df[
            (dependency_df["dataset"] == dataset)
            & (dependency_df["variant"] == variant)
        ]

        dependency_values = pd.to_numeric(
            subset["dependency_error"],
            errors="coerce",
        )

        correlation_values = pd.to_numeric(
            subset["mean_absolute_correlation_error"],
            errors="coerce",
        )

        dependency_values = dependency_values[
            np.isfinite(dependency_values)
        ]

        correlation_values = correlation_values[
            np.isfinite(correlation_values)
        ]

        summary_records.append({
            "dataset": dataset,
            "variant": variant,
            "variant_label": VARIANT_REGISTRY[
                variant
            ]["variant_label"],
            "dependency_guidance_enabled": VARIANT_REGISTRY[
                variant
            ]["dependency_guidance"],
            "mean_dependency_error": (
                float(dependency_values.mean())
                if len(dependency_values) > 0
                else np.nan
            ),
            "std_dependency_error": (
                float(dependency_values.std(ddof=1))
                if len(dependency_values) > 1
                else np.nan
            ),
            "mean_correlation_error": (
                float(correlation_values.mean())
                if len(correlation_values) > 0
                else np.nan
            ),
            "valid_runs": int(len(dependency_values)),
        })

dependency_summary = pd.DataFrame(
    summary_records,
    columns=SUMMARY_COLUMNS,
)

# -----------------------------------------------------------------------------
# 6. SAVE SUMMARY
# -----------------------------------------------------------------------------

dependency_summary.to_csv(
    DEPENDENCY_SUMMARY_PATH,
    index=False,
)

# -----------------------------------------------------------------------------
# 7. VALIDATE OUTPUTS
# -----------------------------------------------------------------------------

assert list(dependency_df.columns) == DEPENDENCY_COLUMNS

assert list(dependency_summary.columns) == SUMMARY_COLUMNS

assert len(dependency_summary) == (
    len(DATASETS) * len(VARIANTS)
)

assert DEPENDENCY_OUTPUT_PATH.exists()
assert DEPENDENCY_SUMMARY_PATH.exists()

total_valid_runs = int(
    dependency_summary["valid_runs"].sum()
)

dependency_status = (
    "PASS"
    if total_valid_runs == EXPECTED_RECORDS
    else "INCOMPLETE"
    if total_valid_runs > 0
    else "BLOCKED"
)

# -----------------------------------------------------------------------------
# 8. DISPLAY RESULTS
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("NOTEBOOK 17 — SECTION 7 DEPENDENCY-GUIDANCE ABLATION")
print("=" * 100)

print("Section 6 fidelity records:", len(fidelity_input))
print("Dependency records:", len(dependency_df))
print("Expected experiment records:", EXPECTED_RECORDS)
print("Valid dependency measurements:", total_valid_runs)

print("\nDependency summary:")

print(
    dependency_summary.to_string(index=False)
)

print("\nComponent results:", DEPENDENCY_OUTPUT_PATH)
print("Summary results:", DEPENDENCY_SUMMARY_PATH)

print("\nDependency output schema validation: PASS")
print("Summary coverage validation: PASS")
print("SECTION 7 STATUS:", dependency_status)

if dependency_status == "BLOCKED":

    print(
        "Dependency evaluation is blocked because Section 6 "
        "has no available fidelity measurements."
    )

elif dependency_status == "INCOMPLETE":

    print(
        "Some dependency measurements are available, but the "
        "complete experiment set has not been evaluated."
    )

else:

    print(
        "All expected dependency measurements are available."
    )

print("=" * 100)

gc.collect()



NOTEBOOK 17 — SECTION 7 DEPENDENCY-GUIDANCE ABLATION
Section 6 fidelity records: 0
Dependency records: 0
Expected experiment records: 60
Valid dependency measurements: 0

Dependency summary:
       dataset                  variant              variant_label  dependency_guidance_enabled  mean_dependency_error  std_dependency_error  mean_correlation_error  valid_runs
  adult_income                 gan_only                   GAN-only                        False                    NaN                   NaN                     NaN           0
  adult_income statistical_guidance_gan Statistical Guidance + GAN                         True                    NaN                   NaN                     NaN           0
  adult_income                   gan_dp GAN + Differential Privacy                        False                    NaN                   NaN                     NaN           0
  adult_income              full_sppgan           Complete SPP-GAN                         True     

32

In [35]:

# =============================================================================
# SECTION 8 — DISTRIBUTION-GUIDANCE ABLATION
# RAM-COMPATIBLE: USE SECTION 6 FIDELITY RESULTS
# =============================================================================

import gc
import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# 1. DEFINE OUTPUT PATHS AND METRICS
# -----------------------------------------------------------------------------

DISTRIBUTION_OUTPUT_PATH = (
    NB17_DIRS["results"] /
    "distribution_guidance_ablation.csv"
)

DISTRIBUTION_SUMMARY_PATH = (
    NB17_DIRS["tables"] /
    "distribution_guidance_summary.csv"
)

distribution_columns = [
    "mean_ks_statistic",
    "mean_wasserstein_distance",
    "mean_moment_mean_error",
    "mean_moment_std_error",
    "mean_js_distance",
    "mean_total_variation_distance",
]

DISTRIBUTION_COLUMNS = [
    "dataset",
    "repetition_id",
    "variant",
    "distribution_guidance_enabled",
    *distribution_columns,
]

SUMMARY_COLUMNS = [
    "dataset",
    "variant",
    "variant_label",
    "distribution_guidance_enabled",
    *[
        f"{metric}_{stat}"
        for metric in distribution_columns
        for stat in ["mean", "std", "count"]
    ],
]

# -----------------------------------------------------------------------------
# 2. VALIDATE SECTION 6 INPUT
# -----------------------------------------------------------------------------

if "fidelity_df" not in globals():
    raise RuntimeError(
        "Section 6 fidelity results are unavailable. Run Section 6 first."
    )

if "VARIANT_REGISTRY" not in globals():
    raise RuntimeError(
        "Variant registry unavailable. Run Section 3 first."
    )

required_columns = [
    "dataset",
    "repetition_id",
    "variant",
    *distribution_columns,
]

missing_columns = [
    column
    for column in required_columns
    if column not in fidelity_df.columns
]

if missing_columns and fidelity_df.empty:

    print(
        "WARNING: No fidelity measurements are available. "
        "Creating schema-compatible empty distribution outputs."
    )

    fidelity_input = pd.DataFrame(
        columns=required_columns
    )

elif missing_columns:

    raise ValueError(
        "Section 6 fidelity results have missing columns: "
        f"{missing_columns}"
    )

else:

    fidelity_input = fidelity_df.copy()

# -----------------------------------------------------------------------------
# 3. EXTRACT DISTRIBUTION-GUIDANCE RESULTS
# -----------------------------------------------------------------------------

distribution_records = []

for _, row in fidelity_input.iterrows():

    variant = str(row["variant"])

    if variant not in VARIANT_REGISTRY:
        raise ValueError(
            f"Unregistered ablation variant: {variant}"
        )

    record = {
        "dataset": row["dataset"],
        "repetition_id": row["repetition_id"],
        "variant": variant,
        "distribution_guidance_enabled": (
            VARIANT_REGISTRY[variant]["distribution_guidance"]
        ),
    }

    for column in distribution_columns:
        record[column] = pd.to_numeric(
            pd.Series([row[column]]),
            errors="coerce",
        ).iloc[0]

    distribution_records.append(record)

distribution_df = pd.DataFrame(
    distribution_records,
    columns=DISTRIBUTION_COLUMNS,
)

# -----------------------------------------------------------------------------
# 4. SAVE COMPONENT-LEVEL RESULTS
# -----------------------------------------------------------------------------

distribution_df.to_csv(
    DISTRIBUTION_OUTPUT_PATH,
    index=False,
)

# -----------------------------------------------------------------------------
# 5. GENERATE COMPLETE DATASET-VARIANT SUMMARY
# -----------------------------------------------------------------------------

summary_records = []

for dataset in DATASETS:

    for variant in VARIANTS:

        subset = distribution_df[
            (distribution_df["dataset"] == dataset)
            & (distribution_df["variant"] == variant)
        ]

        summary_record = {
            "dataset": dataset,
            "variant": variant,
            "variant_label": VARIANT_REGISTRY[
                variant
            ]["variant_label"],
            "distribution_guidance_enabled": (
                VARIANT_REGISTRY[variant]["distribution_guidance"]
            ),
        }

        for metric in distribution_columns:

            values = pd.to_numeric(
                subset[metric],
                errors="coerce",
            )

            values = values[np.isfinite(values)]

            summary_record[f"{metric}_mean"] = (
                float(values.mean())
                if len(values) > 0
                else np.nan
            )

            summary_record[f"{metric}_std"] = (
                float(values.std(ddof=1))
                if len(values) > 1
                else np.nan
            )

            summary_record[f"{metric}_count"] = int(
                len(values)
            )

        summary_records.append(summary_record)

distribution_summary = pd.DataFrame(
    summary_records,
    columns=SUMMARY_COLUMNS,
)

# -----------------------------------------------------------------------------
# 6. SAVE SUMMARY
# -----------------------------------------------------------------------------

distribution_summary.to_csv(
    DISTRIBUTION_SUMMARY_PATH,
    index=False,
)

# -----------------------------------------------------------------------------
# 7. VALIDATE OUTPUTS
# -----------------------------------------------------------------------------

assert list(distribution_df.columns) == DISTRIBUTION_COLUMNS

assert list(distribution_summary.columns) == SUMMARY_COLUMNS

assert len(distribution_summary) == (
    len(DATASETS) * len(VARIANTS)
)

assert distribution_df["variant"].isin(VARIANTS).all()

assert DISTRIBUTION_OUTPUT_PATH.exists()
assert DISTRIBUTION_SUMMARY_PATH.exists()

total_valid_measurements = sum(
    distribution_summary[f"{metric}_count"].sum()
    for metric in distribution_columns
)

distribution_status = (
    "PASS"
    if len(distribution_df) == EXPECTED_RECORDS
    else "INCOMPLETE"
    if len(distribution_df) > 0
    else "BLOCKED"
)

# -----------------------------------------------------------------------------
# 8. DISPLAY RESULTS
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("NOTEBOOK 17 — SECTION 8 DISTRIBUTION-GUIDANCE ABLATION")
print("=" * 100)

print("Section 6 fidelity records:", len(fidelity_input))
print("Distribution records:", len(distribution_df))
print("Expected experiment records:", EXPECTED_RECORDS)
print("Dataset-variant summary rows:", len(distribution_summary))
print("Valid metric measurements:", int(total_valid_measurements))

print("\nDistribution summary:")

display_columns = [
    "dataset",
    "variant",
    "distribution_guidance_enabled",
    "mean_ks_statistic_mean",
    "mean_wasserstein_distance_mean",
    "mean_js_distance_mean",
    "mean_total_variation_distance_mean",
]

print(
    distribution_summary[display_columns].to_string(index=False)
)

print("\nComponent results:", DISTRIBUTION_OUTPUT_PATH)
print("Summary results:", DISTRIBUTION_SUMMARY_PATH)

print("\nComponent schema validation: PASS")
print("Summary coverage validation: PASS")
print("SECTION 8 STATUS:", distribution_status)

if distribution_status == "BLOCKED":

    print(
        "Distribution evaluation is blocked because Section 6 "
        "contains no fidelity measurements."
    )

elif distribution_status == "INCOMPLETE":

    print(
        "Distribution measurements are incomplete. "
        "Complete the missing experiments before interpreting results."
    )

else:

    print(
        "All expected distribution-guidance evaluations are available."
    )

print("=" * 100)

gc.collect()



NOTEBOOK 17 — SECTION 8 DISTRIBUTION-GUIDANCE ABLATION
Section 6 fidelity records: 0
Distribution records: 0
Expected experiment records: 60
Dataset-variant summary rows: 12
Valid metric measurements: 0

Distribution summary:
       dataset                  variant  distribution_guidance_enabled  mean_ks_statistic_mean  mean_wasserstein_distance_mean  mean_js_distance_mean  mean_total_variation_distance_mean
  adult_income                 gan_only                          False                     NaN                             NaN                    NaN                                 NaN
  adult_income statistical_guidance_gan                           True                     NaN                             NaN                    NaN                                 NaN
  adult_income                   gan_dp                          False                     NaN                             NaN                    NaN                                 NaN
  adult_income              f

32

In [37]:

# =============================================================================
# SECTION 9 — DIFFERENTIAL PRIVACY ABLATION
# FORMAL PRIVACY EVIDENCE VALIDATION
# =============================================================================

import gc
import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# 1. CONFIGURATION
# -----------------------------------------------------------------------------

TARGET_EPSILON = 5.0
DP_ACCOUNTANT = "RDP"
PRIVATE_COMPONENT = "critic"

DP_OUTPUT_PATH = (
    NB17_DIRS["results"] /
    "dp_ablation_results.csv"
)

DP_SUMMARY_PATH = (
    NB17_DIRS["tables"] /
    "dp_ablation_summary.csv"
)

DP_COLUMNS = [
    "dataset",
    "repetition_id",
    "variant",
    "dp_applied",
    "achieved_epsilon",
    "delta",
    "privacy_scope",
    "accountant",
    "private_component",
    "target_epsilon",
    "epsilon_within_target",
    "formal_dp_evidence_available",
    "privacy_evidence_status",
]

DP_SUMMARY_COLUMNS = [
    "dataset",
    "variant",
    "dp_applied",
    "configured_dp_runs",
    "recorded_epsilon_runs",
    "recorded_delta_runs",
    "formal_evidence_runs",
    "mean_achieved_epsilon",
    "std_achieved_epsilon",
    "mean_delta",
    "epsilon_within_target_runs",
]

# -----------------------------------------------------------------------------
# 2. VALIDATE EXPERIMENT REGISTRY
# -----------------------------------------------------------------------------

if "registry_df" not in globals():
    raise RuntimeError(
        "Experiment registry unavailable. Run Section 4 first."
    )

required_columns = [
    "dataset",
    "repetition_id",
    "variant",
]

missing_columns = [
    column for column in required_columns
    if column not in registry_df.columns
]

if missing_columns:
    raise ValueError(
        f"Experiment registry missing columns: {missing_columns}"
    )

for column in [
    "achieved_epsilon",
    "delta",
    "privacy_scope",
    "accountant",
    "private_component",
]:
    if column not in registry_df.columns:
        registry_df[column] = np.nan

# -----------------------------------------------------------------------------
# 3. NORMALIZE PRIVACY METADATA
# -----------------------------------------------------------------------------

dp_input = registry_df.copy()

dp_input["achieved_epsilon"] = pd.to_numeric(
    dp_input["achieved_epsilon"],
    errors="coerce",
)

dp_input["delta"] = pd.to_numeric(
    dp_input["delta"],
    errors="coerce",
)

# -----------------------------------------------------------------------------
# 4. EXTRACT DP EVIDENCE
# -----------------------------------------------------------------------------

dp_records = []

for _, row in dp_input.iterrows():

    variant = str(row["variant"])

    if variant not in VARIANT_REGISTRY:
        raise ValueError(
            f"Unregistered variant found: {variant}"
        )

    dp_enabled = bool(
        VARIANT_REGISTRY[variant]["differential_privacy"]
    )

    epsilon = row["achieved_epsilon"]
    delta = row["delta"]

    privacy_scope = (
        str(row["privacy_scope"]).strip()
        if pd.notna(row["privacy_scope"])
        else ""
    )

    accountant = (
        str(row["accountant"]).strip()
        if pd.notna(row["accountant"])
        else ""
    )

    private_component = (
        str(row["private_component"]).strip()
        if pd.notna(row["private_component"])
        else ""
    )

    epsilon_valid = (
        pd.notna(epsilon)
        and np.isfinite(epsilon)
        and epsilon > 0
    )

    delta_valid = (
        pd.notna(delta)
        and np.isfinite(delta)
        and 0 < delta < 1
    )

    scope_valid = bool(privacy_scope)
    accountant_valid = bool(accountant)
    component_valid = bool(private_component)

    formal_evidence = (
        dp_enabled
        and epsilon_valid
        and delta_valid
        and scope_valid
        and accountant_valid
        and component_valid
    )

    epsilon_within_target = (
        bool(epsilon <= TARGET_EPSILON)
        if dp_enabled and epsilon_valid
        else False
    )

    if not dp_enabled:
        evidence_status = "NOT_APPLICABLE"

    elif formal_evidence:
        evidence_status = "RECORDED_EVIDENCE"

    else:
        evidence_status = "MISSING_OR_INCOMPLETE_EVIDENCE"

    dp_records.append({
        "dataset": str(row["dataset"]),
        "repetition_id": row["repetition_id"],
        "variant": variant,
        "dp_applied": dp_enabled,
        "achieved_epsilon": epsilon,
        "delta": delta,
        "privacy_scope": privacy_scope or np.nan,
        "accountant": accountant or np.nan,
        "private_component": private_component or np.nan,
        "target_epsilon": TARGET_EPSILON,
        "epsilon_within_target": epsilon_within_target,
        "formal_dp_evidence_available": formal_evidence,
        "privacy_evidence_status": evidence_status,
    })

dp_df = pd.DataFrame(
    dp_records,
    columns=DP_COLUMNS,
)

# -----------------------------------------------------------------------------
# 5. SAVE EXPERIMENT-LEVEL DP RESULTS
# -----------------------------------------------------------------------------

dp_df.to_csv(
    DP_OUTPUT_PATH,
    index=False,
)

# -----------------------------------------------------------------------------
# 6. GENERATE COMPLETE DATASET-VARIANT SUMMARY
# -----------------------------------------------------------------------------

dp_summary_records = []

for dataset in DATASETS:

    for variant in VARIANTS:

        subset = dp_df[
            (dp_df["dataset"] == dataset)
            & (dp_df["variant"] == variant)
        ]

        dp_enabled = bool(
            VARIANT_REGISTRY[variant]["differential_privacy"]
        )

        epsilon_values = pd.to_numeric(
            subset["achieved_epsilon"],
            errors="coerce",
        )

        delta_values = pd.to_numeric(
            subset["delta"],
            errors="coerce",
        )

        valid_epsilon = epsilon_values[
            np.isfinite(epsilon_values) & (epsilon_values > 0)
        ]

        valid_delta = delta_values[
            np.isfinite(delta_values)
            & (delta_values > 0)
            & (delta_values < 1)
        ]

        dp_summary_records.append({
            "dataset": dataset,
            "variant": variant,
            "dp_applied": dp_enabled,

            "configured_dp_runs": (
                int(subset["dp_applied"].sum())
            ),

            "recorded_epsilon_runs": int(
                len(valid_epsilon)
            ),

            "recorded_delta_runs": int(
                len(valid_delta)
            ),

            "formal_evidence_runs": int(
                subset["formal_dp_evidence_available"].sum()
            ),

            "mean_achieved_epsilon": (
                float(valid_epsilon.mean())
                if len(valid_epsilon) > 0
                else np.nan
            ),

            "std_achieved_epsilon": (
                float(valid_epsilon.std(ddof=1))
                if len(valid_epsilon) > 1
                else np.nan
            ),

            "mean_delta": (
                float(valid_delta.mean())
                if len(valid_delta) > 0
                else np.nan
            ),

            "epsilon_within_target_runs": int(
                subset["epsilon_within_target"].sum()
            ),
        })

dp_summary = pd.DataFrame(
    dp_summary_records,
    columns=DP_SUMMARY_COLUMNS,
)

# -----------------------------------------------------------------------------
# 7. SAVE SUMMARY
# -----------------------------------------------------------------------------

dp_summary.to_csv(
    DP_SUMMARY_PATH,
    index=False,
)

# -----------------------------------------------------------------------------
# 8. VALIDATE EXPERIMENT COVERAGE
# -----------------------------------------------------------------------------

expected_dp_runs = (
    len(DATASETS)
    * len(REPETITIONS)
    * sum(
        VARIANT_REGISTRY[v]["differential_privacy"]
        for v in VARIANTS
    )
)

expected_non_dp_runs = (
    EXPECTED_RECORDS - expected_dp_runs
)

configured_dp_runs = int(dp_df["dp_applied"].sum())

formal_evidence_runs = int(
    dp_df["formal_dp_evidence_available"].sum()
)

assert len(dp_df) == EXPECTED_RECORDS

assert len(dp_summary) == (
    len(DATASETS) * len(VARIANTS)
)

assert dp_df["variant"].isin(VARIANTS).all()

assert configured_dp_runs == expected_dp_runs

assert expected_dp_runs == 30
assert expected_non_dp_runs == 30

assert DP_OUTPUT_PATH.exists()
assert DP_SUMMARY_PATH.exists()

# -----------------------------------------------------------------------------
# 9. DETERMINE SECTION STATUS
# -----------------------------------------------------------------------------

if formal_evidence_runs == expected_dp_runs:
    dp_status = "PASS"

elif formal_evidence_runs > 0:
    dp_status = "INCOMPLETE"

else:
    dp_status = "BLOCKED"

# -----------------------------------------------------------------------------
# 10. DISPLAY RESULTS
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("NOTEBOOK 17 — SECTION 9 DIFFERENTIAL PRIVACY ABLATION")
print("=" * 100)

print("Total experiment records:", len(dp_df))
print("Expected DP-configured runs:", expected_dp_runs)
print("Configured DP runs:", configured_dp_runs)
print("Expected non-DP runs:", expected_non_dp_runs)

print(
    "Recorded achieved epsilon values:",
    int(dp_df["achieved_epsilon"].notna().sum())
)

print(
    "Recorded delta values:",
    int(dp_df["delta"].notna().sum())
)

print("Formal DP evidence records:", formal_evidence_runs)

print("\nDP summary:")
print(dp_summary.to_string(index=False))

print("\nExperiment results:", DP_OUTPUT_PATH)
print("Summary results:", DP_SUMMARY_PATH)

print("\nRegistry coverage validation: PASS")
print("DP variant configuration validation: PASS")
print("Formal privacy evidence validation:", dp_status)

print("\nSECTION 9 STATUS:", dp_status)

if dp_status == "BLOCKED":
    print(
        "No complete formal DP evidence is recorded. "
        "Retrieve actual per-run privacy accountant outputs "
        "before claiming an achieved privacy guarantee."
    )

elif dp_status == "INCOMPLETE":
    print(
        "Formal DP evidence is available for only some configured "
        "DP experiments. Complete the missing evidence."
    )

else:
    print(
        "Formal DP evidence is recorded for all configured DP runs. "
        "Review the accountant outputs and privacy scope before "
        "making formal privacy claims."
    )

print(
    "\nIMPORTANT: Configured DP variants, achieved epsilon/delta, "
    "and empirical disclosure-risk measurements are distinct "
    "experimental results and must be reported separately."
)

print("=" * 100)

gc.collect()



NOTEBOOK 17 — SECTION 9 DIFFERENTIAL PRIVACY ABLATION
Total experiment records: 60
Expected DP-configured runs: 30
Configured DP runs: 30
Expected non-DP runs: 30
Recorded achieved epsilon values: 0
Recorded delta values: 0
Formal DP evidence records: 0

DP summary:
       dataset                  variant  dp_applied  configured_dp_runs  recorded_epsilon_runs  recorded_delta_runs  formal_evidence_runs  mean_achieved_epsilon  std_achieved_epsilon  mean_delta  epsilon_within_target_runs
  adult_income                 gan_only       False                   0                      0                    0                     0                    NaN                   NaN         NaN                           0
  adult_income statistical_guidance_gan       False                   0                      0                    0                     0                    NaN                   NaN         NaN                           0
  adult_income                   gan_dp        True            

0

In [39]:
# =============================================================================
# SECTION 10 — FULL SPP-GAN EVALUATION
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np

# -----------------------------------------------------------------------------
# 1. LOAD OPTIONAL EVALUATION ARTIFACTS
# -----------------------------------------------------------------------------

def load_optional_artifact(candidates, artifact_name):

    for candidate in candidates:
        path = Path(candidate)

        if path.exists():
            try:
                df = pd.read_csv(path, low_memory=True)

                print(
                    f"{artifact_name} loaded:",
                    path,
                    "| Rows:", len(df)
                )

                return df

            except Exception as exc:
                print(
                    f"{artifact_name} read error:",
                    path,
                    "|", exc
                )

    print(f"{artifact_name}: not available.")
    return pd.DataFrame()


utility_df = load_optional_artifact(
    UTILITY_CANDIDATES,
    "TSTR utility"
)

privacy_df = load_optional_artifact(
    PRIVACY_CANDIDATES,
    "Privacy/disclosure"
)

# -----------------------------------------------------------------------------
# 2. DEFINE JOIN KEYS AND EXPECTED EXPERIMENT GRID
# -----------------------------------------------------------------------------

JOIN_KEYS = [
    "dataset",
    "repetition_id",
    "variant"
]

if "registry_df" not in globals() or registry_df.empty:
    raise RuntimeError(
        "Experiment registry is unavailable. "
        "Run Section 4 before Section 10."
    )

missing_registry_keys = [
    col for col in JOIN_KEYS
    if col not in registry_df.columns
]

if missing_registry_keys:
    raise ValueError(
        "Experiment registry is missing join keys: "
        f"{missing_registry_keys}"
    )

base_df = registry_df.copy()

base_df["dataset"] = base_df["dataset"].astype(str)
base_df["variant"] = base_df["variant"].astype(str)

base_df["repetition_id"] = pd.to_numeric(
    base_df["repetition_id"],
    errors="coerce"
)

if base_df["repetition_id"].isna().any():
    raise ValueError(
        "Experiment registry contains invalid repetition_id values."
    )

if base_df.duplicated(JOIN_KEYS).any():
    duplicates = base_df.loc[
        base_df.duplicated(JOIN_KEYS, keep=False),
        JOIN_KEYS
    ]

    raise ValueError(
        "Experiment registry contains duplicate experiment keys:\n"
        f"{duplicates.to_string(index=False)}"
    )

# -----------------------------------------------------------------------------
# 3. NORMALIZE EVALUATION TABLES
# -----------------------------------------------------------------------------

def normalize_evaluation_table(df, name):

    if df.empty:
        print(f"{name}: empty or unavailable.")
        return pd.DataFrame()

    missing = [
        col for col in JOIN_KEYS
        if col not in df.columns
    ]

    if missing:
        print(
            f"{name}: missing join keys {missing}. "
            "Artifact excluded."
        )
        return pd.DataFrame()

    normalized = df.copy()

    normalized["dataset"] = (
        normalized["dataset"].astype(str)
    )

    normalized["variant"] = (
        normalized["variant"].astype(str)
    )

    normalized["repetition_id"] = pd.to_numeric(
        normalized["repetition_id"],
        errors="coerce"
    )

    normalized = normalized.dropna(
        subset=JOIN_KEYS
    )

    if normalized.duplicated(JOIN_KEYS).any():

        duplicates = normalized.loc[
            normalized.duplicated(JOIN_KEYS, keep=False),
            JOIN_KEYS
        ]

        raise ValueError(
            f"{name} contains duplicate experiment keys. "
            "Resolve duplicates before merging:\n"
            f"{duplicates.to_string(index=False)}"
        )

    return normalized


fidelity_join = normalize_evaluation_table(
    fidelity_df,
    "Fidelity"
)

utility_join = normalize_evaluation_table(
    utility_df,
    "Utility"
)

privacy_join = normalize_evaluation_table(
    privacy_df,
    "Privacy"
)

# -----------------------------------------------------------------------------
# 4. MERGE FIDELITY, UTILITY AND PRIVACY RESULTS
# -----------------------------------------------------------------------------

evaluation_df = base_df.copy()

artifact_status = {
    "fidelity": not fidelity_join.empty,
    "utility": not utility_join.empty,
    "privacy": not privacy_join.empty
}


def merge_evaluation_artifact(
    base,
    artifact,
    artifact_name
):

    if artifact.empty:
        print(f"{artifact_name}: no records merged.")
        return base

    artifact_columns = [
        col for col in artifact.columns
        if col not in JOIN_KEYS
    ]

    # Avoid accidental overwriting of registry fields.
    rename_map = {
        col: f"{col}_{artifact_name.lower()}"
        for col in artifact_columns
        if col in base.columns
    }

    artifact = artifact.rename(columns=rename_map)

    merged = base.merge(
        artifact,
        on=JOIN_KEYS,
        how="left",
        validate="one_to_one"
    )

    print(
        f"{artifact_name} merged | "
        f"Artifact rows: {len(artifact)} | "
        f"Matched experiments: "
        f"{merged[artifact_columns[0]].notna().sum() if artifact_columns else 0}"
    )

    return merged


evaluation_df = merge_evaluation_artifact(
    evaluation_df,
    fidelity_join,
    "Fidelity"
)

evaluation_df = merge_evaluation_artifact(
    evaluation_df,
    utility_join,
    "Utility"
)

evaluation_df = merge_evaluation_artifact(
    evaluation_df,
    privacy_join,
    "Privacy"
)

# -----------------------------------------------------------------------------
# 5. ATTACH VARIANT LABELS
# -----------------------------------------------------------------------------

evaluation_df["variant_label"] = (
    evaluation_df["variant"].map(
        lambda variant: VARIANT_REGISTRY.get(
            variant, {}
        ).get("variant_label", variant)
    )
)

# -----------------------------------------------------------------------------
# 6. VALIDATE EXPERIMENT COVERAGE
# -----------------------------------------------------------------------------

expected_records = len(base_df)
combined_records = len(evaluation_df)

if combined_records != expected_records:
    raise AssertionError(
        "Combined evaluation row count does not match "
        f"the experiment registry: {combined_records} vs "
        f"{expected_records}."
    )

if evaluation_df.duplicated(JOIN_KEYS).any():
    raise AssertionError(
        "Duplicate experiment keys detected after merging."
    )

coverage_rows = []

for artifact_name, artifact_df in [
    ("Fidelity", fidelity_join),
    ("Utility", utility_join),
    ("Privacy", privacy_join)
]:

    if artifact_df.empty:
        matched = 0
    else:
        matched = (
            evaluation_df[JOIN_KEYS]
            .merge(
                artifact_df[JOIN_KEYS],
                on=JOIN_KEYS,
                how="inner"
            )
            .shape[0]
        )

    coverage_rows.append({
        "artifact": artifact_name,
        "expected_experiments": expected_records,
        "available_records": len(artifact_df),
        "matched_experiments": matched,
        "missing_experiments": expected_records - matched,
        "coverage_complete": matched == expected_records
    })


coverage_df = pd.DataFrame(coverage_rows)

# -----------------------------------------------------------------------------
# 7. SAVE FULL EVALUATION ARTIFACTS
# -----------------------------------------------------------------------------

results_path = (
    NB17_DIRS["results"] /
    "ablation_results.csv"
)

coverage_path = (
    NB17_DIRS["validation"] /
    "ablation_evaluation_coverage.csv"
)

evaluation_df.to_csv(
    results_path,
    index=False
)

coverage_df.to_csv(
    coverage_path,
    index=False
)

# -----------------------------------------------------------------------------
# 8. FINAL SECTION VALIDATION
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("SECTION 10 — FULL SPP-GAN EVALUATION")
print("=" * 80)

print("Expected experiment records:", expected_records)
print("Combined evaluation rows:", combined_records)
print("Combined evaluation columns:", len(evaluation_df.columns))

print(
    "Datasets:",
    evaluation_df["dataset"].nunique()
)

print(
    "Variants:",
    evaluation_df["variant"].nunique()
)

print("\nArtifact availability:")
for name, available in artifact_status.items():
    print(f"{name.title()}: {available}")

print("\nEvaluation coverage:")
print(coverage_df.to_string(index=False))

print("\nCombined evaluation preview:")
print(evaluation_df.head())

print("\nSaved:")
print("Evaluation:", results_path)
print("Coverage:", coverage_path)

if all(artifact_status.values()) and coverage_df["coverage_complete"].all():
    SECTION_10_STATUS = "PASS"

elif combined_records == expected_records:
    SECTION_10_STATUS = "INCOMPLETE"

else:
    SECTION_10_STATUS = "BLOCKED"

print("\nSECTION 10 STATUS:", SECTION_10_STATUS)

print("=" * 80)

TSTR utility: not available.
Privacy/disclosure: not available.
Fidelity: empty or unavailable.
Utility: empty or unavailable.
Privacy: empty or unavailable.
Fidelity: no records merged.
Utility: no records merged.
Privacy: no records merged.

SECTION 10 — FULL SPP-GAN EVALUATION
Expected experiment records: 60
Combined evaluation rows: 60
Combined evaluation columns: 24
Datasets: 3
Variants: 4

Artifact availability:
Fidelity: False
Utility: False
Privacy: False

Evaluation coverage:
artifact  expected_experiments  available_records  matched_experiments  missing_experiments  coverage_complete
Fidelity                    60                  0                    0                   60              False
 Utility                    60                  0                    0                   60              False
 Privacy                    60                  0                    0                   60              False

Combined evaluation preview:
        dataset  repetition_id      

In [41]:
# =============================================================================
# SECTION 11 — FIDELITY COMPARISON
# =============================================================================

import pandas as pd
import numpy as np

FIDELITY_METRICS = [
    "mean_ks_statistic",
    "mean_wasserstein_distance",
    "mean_moment_mean_error",
    "mean_moment_std_error",
    "mean_js_distance",
    "mean_total_variation_distance",
    "correlation_frobenius_error",
    "mean_absolute_correlation_error"
]

GROUP_KEYS = ["dataset", "variant"]

# -----------------------------------------------------------------------------
# 1. VALIDATE INPUT
# -----------------------------------------------------------------------------

if "fidelity_df" not in globals():
    raise RuntimeError(
        "fidelity_df is unavailable. Run Section 6 before Section 11."
    )

missing_keys = [
    col for col in GROUP_KEYS
    if col not in fidelity_df.columns
]

if missing_keys:
    raise ValueError(
        f"Fidelity results are missing grouping columns: {missing_keys}"
    )

available_fidelity_metrics = [
    col for col in FIDELITY_METRICS
    if col in fidelity_df.columns
]

if not available_fidelity_metrics:
    raise ValueError(
        "No expected fidelity metrics were found in fidelity_df."
    )

# -----------------------------------------------------------------------------
# 2. NORMALIZE METRIC VALUES
# -----------------------------------------------------------------------------

fidelity_input = fidelity_df.copy()

for col in available_fidelity_metrics:
    fidelity_input[col] = pd.to_numeric(
        fidelity_input[col],
        errors="coerce"
    )

# -----------------------------------------------------------------------------
# 3. BUILD COMPLETE DATASET × VARIANT GRID
# -----------------------------------------------------------------------------

expected_grid = pd.MultiIndex.from_product(
    [DATASETS, VARIANTS],
    names=GROUP_KEYS
).to_frame(index=False)

if fidelity_input.empty:
    fidelity_summary = expected_grid.copy()

    for metric in available_fidelity_metrics:
        fidelity_summary[f"{metric}_mean"] = np.nan
        fidelity_summary[f"{metric}_std"] = np.nan
        fidelity_summary[f"{metric}_count"] = 0

else:
    grouped = (
        fidelity_input
        .groupby(GROUP_KEYS)[available_fidelity_metrics]
        .agg(["mean", "std", "count"])
    )

    grouped.columns = [
        f"{metric}_{stat}"
        for metric, stat in grouped.columns
    ]

    grouped = grouped.reset_index()

    fidelity_summary = expected_grid.merge(
        grouped,
        on=GROUP_KEYS,
        how="left",
        validate="one_to_one"
    )

    count_columns = [
        col for col in fidelity_summary.columns
        if col.endswith("_count")
    ]

    fidelity_summary[count_columns] = (
        fidelity_summary[count_columns]
        .fillna(0)
        .astype(int)
    )

# -----------------------------------------------------------------------------
# 4. OVERALL VARIANT COMPARISON
# -----------------------------------------------------------------------------

if fidelity_input.empty:
    fidelity_overall = pd.DataFrame({
        "variant": VARIANTS
    })

    for metric in available_fidelity_metrics:
        fidelity_overall[f"{metric}_mean"] = np.nan
        fidelity_overall[f"{metric}_std"] = np.nan
        fidelity_overall[f"{metric}_count"] = 0

else:
    overall_grouped = (
        fidelity_input
        .groupby("variant")[available_fidelity_metrics]
        .agg(["mean", "std", "count"])
    )

    overall_grouped.columns = [
        f"{metric}_{stat}"
        for metric, stat in overall_grouped.columns
    ]

    fidelity_overall = (
        pd.DataFrame({"variant": VARIANTS})
        .merge(
            overall_grouped.reset_index(),
            on="variant",
            how="left",
            validate="one_to_one"
        )
    )

    count_columns = [
        col for col in fidelity_overall.columns
        if col.endswith("_count")
    ]

    fidelity_overall[count_columns] = (
        fidelity_overall[count_columns]
        .fillna(0)
        .astype(int)
    )

# -----------------------------------------------------------------------------
# 5. SAVE COMPARISON TABLES
# -----------------------------------------------------------------------------

comparison_path = (
    NB17_DIRS["tables"] /
    "fidelity_comparison.csv"
)

overall_path = (
    NB17_DIRS["tables"] /
    "fidelity_overall_comparison.csv"
)

fidelity_summary.to_csv(
    comparison_path,
    index=False
)

fidelity_overall.to_csv(
    overall_path,
    index=False
)

# -----------------------------------------------------------------------------
# 6. VALIDATION AND STATUS
# -----------------------------------------------------------------------------

expected_summary_rows = len(DATASETS) * len(VARIANTS)

assert len(fidelity_summary) == expected_summary_rows
assert len(fidelity_overall) == len(VARIANTS)

valid_measurements = int(
    fidelity_overall[
        [
            f"{metric}_count"
            for metric in available_fidelity_metrics
        ]
    ].to_numpy().sum()
)

SECTION_11_STATUS = (
    "PASS"
    if valid_measurements > 0
    else "BLOCKED"
)

print("FIDELITY COMPARISON")
print("=" * 80)

print("Expected dataset-variant groups:", expected_summary_rows)
print("Fidelity input records:", len(fidelity_input))
print("Available fidelity metrics:", len(available_fidelity_metrics))
print("Valid metric measurements:", valid_measurements)

print("\nDataset-level fidelity comparison:")
print(fidelity_summary.round(5).to_string(index=False))

print("\nOverall variant comparison:")
print(fidelity_overall.round(5).to_string(index=False))

print("\nSaved:")
print("Dataset comparison:", comparison_path)
print("Overall comparison:", overall_path)

print("\nSECTION 11 STATUS:", SECTION_11_STATUS)

FIDELITY COMPARISON
Expected dataset-variant groups: 12
Fidelity input records: 0
Available fidelity metrics: 8
Valid metric measurements: 0

Dataset-level fidelity comparison:
       dataset                  variant  mean_ks_statistic_mean  mean_ks_statistic_std  mean_ks_statistic_count  mean_wasserstein_distance_mean  mean_wasserstein_distance_std  mean_wasserstein_distance_count  mean_moment_mean_error_mean  mean_moment_mean_error_std  mean_moment_mean_error_count  mean_moment_std_error_mean  mean_moment_std_error_std  mean_moment_std_error_count  mean_js_distance_mean  mean_js_distance_std  mean_js_distance_count  mean_total_variation_distance_mean  mean_total_variation_distance_std  mean_total_variation_distance_count  correlation_frobenius_error_mean  correlation_frobenius_error_std  correlation_frobenius_error_count  mean_absolute_correlation_error_mean  mean_absolute_correlation_error_std  mean_absolute_correlation_error_count
  adult_income                 gan_only            

In [43]:
# =============================================================================
# SECTION 12 — UTILITY COMPARISON
# =============================================================================

import pandas as pd
import numpy as np

UTILITY_METRICS = [
    "accuracy",
    "macro_f1",
    "weighted_f1"
]

GROUP_KEYS = ["dataset", "variant"]

# -----------------------------------------------------------------------------
# 1. VALIDATE AND NORMALIZE UTILITY INPUT
# -----------------------------------------------------------------------------

if "utility_df" not in globals():
    utility_df = pd.DataFrame()

utility_input = utility_df.copy()

if not utility_input.empty:

    missing_keys = [
        col for col in GROUP_KEYS + ["repetition_id"]
        if col not in utility_input.columns
    ]

    if missing_keys:
        print(
            "Utility artifact excluded. Missing columns:",
            missing_keys
        )
        utility_input = pd.DataFrame()

    else:
        utility_input["dataset"] = (
            utility_input["dataset"].astype(str)
        )

        utility_input["variant"] = (
            utility_input["variant"].astype(str)
        )

        utility_input["repetition_id"] = pd.to_numeric(
            utility_input["repetition_id"],
            errors="coerce"
        )

        utility_input = utility_input.dropna(
            subset=GROUP_KEYS + ["repetition_id"]
        )

        available_utility_metrics = [
            col for col in UTILITY_METRICS
            if col in utility_input.columns
        ]

        for col in available_utility_metrics:
            utility_input[col] = pd.to_numeric(
                utility_input[col],
                errors="coerce"
            )

        if not available_utility_metrics:
            print(
                "Utility artifact contains no recognized utility metrics."
            )
            utility_input = pd.DataFrame()

else:
    available_utility_metrics = []

# -----------------------------------------------------------------------------
# 2. DEFINE COMPLETE DATASET × VARIANT GRID
# -----------------------------------------------------------------------------

expected_grid = pd.MultiIndex.from_product(
    [DATASETS, VARIANTS],
    names=GROUP_KEYS
).to_frame(index=False)

# -----------------------------------------------------------------------------
# 3. DATASET-LEVEL UTILITY COMPARISON
# -----------------------------------------------------------------------------

if not utility_input.empty:

    grouped_utility = (
        utility_input
        .groupby(GROUP_KEYS)[available_utility_metrics]
        .agg(["mean", "std", "count"])
    )

    grouped_utility.columns = [
        f"{metric}_{stat}"
        for metric, stat in grouped_utility.columns
    ]

    utility_comparison = expected_grid.merge(
        grouped_utility.reset_index(),
        on=GROUP_KEYS,
        how="left",
        validate="one_to_one"
    )

else:
    available_utility_metrics = []

    utility_comparison = expected_grid.copy()

# Ensure a stable output schema, including when no utility results exist.
for metric in UTILITY_METRICS:

    for stat in ["mean", "std"]:
        col = f"{metric}_{stat}"

        if col not in utility_comparison.columns:
            utility_comparison[col] = np.nan

    count_col = f"{metric}_count"

    if count_col not in utility_comparison.columns:
        utility_comparison[count_col] = 0

    utility_comparison[count_col] = (
        utility_comparison[count_col]
        .fillna(0)
        .astype(int)
    )

# -----------------------------------------------------------------------------
# 4. OVERALL VARIANT COMPARISON
# -----------------------------------------------------------------------------

utility_overall = pd.DataFrame({
    "variant": VARIANTS
})

if not utility_input.empty:

    overall_grouped = (
        utility_input
        .groupby("variant")[available_utility_metrics]
        .agg(["mean", "std", "count"])
    )

    overall_grouped.columns = [
        f"{metric}_{stat}"
        for metric, stat in overall_grouped.columns
    ]

    utility_overall = utility_overall.merge(
        overall_grouped.reset_index(),
        on="variant",
        how="left",
        validate="one_to_one"
    )

for metric in UTILITY_METRICS:

    for stat in ["mean", "std"]:
        col = f"{metric}_{stat}"

        if col not in utility_overall.columns:
            utility_overall[col] = np.nan

    count_col = f"{metric}_count"

    if count_col not in utility_overall.columns:
        utility_overall[count_col] = 0

    utility_overall[count_col] = (
        utility_overall[count_col]
        .fillna(0)
        .astype(int)
    )

# -----------------------------------------------------------------------------
# 5. SAVE COMPARISON ARTIFACTS
# -----------------------------------------------------------------------------

comparison_path = (
    NB17_DIRS["tables"] /
    "utility_comparison.csv"
)

overall_path = (
    NB17_DIRS["tables"] /
    "utility_overall_comparison.csv"
)

utility_comparison.to_csv(
    comparison_path,
    index=False
)

utility_overall.to_csv(
    overall_path,
    index=False
)

# -----------------------------------------------------------------------------
# 6. VALIDATE AND REPORT STATUS
# -----------------------------------------------------------------------------

expected_groups = len(DATASETS) * len(VARIANTS)

assert len(utility_comparison) == expected_groups
assert len(utility_overall) == len(VARIANTS)

count_columns = [
    f"{metric}_count"
    for metric in UTILITY_METRICS
]

valid_measurements = int(
    utility_overall[count_columns].to_numpy().sum()
)

SECTION_12_STATUS = (
    "PASS"
    if valid_measurements > 0
    else "BLOCKED"
)

print("UTILITY COMPARISON")
print("=" * 80)

print("Utility input records:", len(utility_input))
print("Expected dataset-variant groups:", expected_groups)
print("Available utility metrics:", available_utility_metrics)
print("Valid utility measurements:", valid_measurements)

print("\nDataset-level utility comparison:")
print(utility_comparison.round(5).to_string(index=False))

print("\nOverall utility comparison:")
print(utility_overall.round(5).to_string(index=False))

print("\nSaved:")
print("Dataset comparison:", comparison_path)
print("Overall comparison:", overall_path)

print("\nSECTION 12 STATUS:", SECTION_12_STATUS)
print("=" * 80)

UTILITY COMPARISON
Utility input records: 0
Expected dataset-variant groups: 12
Available utility metrics: []
Valid utility measurements: 0

Dataset-level utility comparison:
       dataset                  variant  accuracy_mean  accuracy_std  accuracy_count  macro_f1_mean  macro_f1_std  macro_f1_count  weighted_f1_mean  weighted_f1_std  weighted_f1_count
  adult_income                 gan_only            NaN           NaN               0            NaN           NaN               0               NaN              NaN                  0
  adult_income statistical_guidance_gan            NaN           NaN               0            NaN           NaN               0               NaN              NaN                  0
  adult_income                   gan_dp            NaN           NaN               0            NaN           NaN               0               NaN              NaN                  0
  adult_income              full_sppgan            NaN           NaN               0     

In [45]:
# =============================================================================
# SECTION 13 — PRIVACY COMPARISON
# =============================================================================

import pandas as pd
import numpy as np

PRIVACY_METRICS = [
    "achieved_epsilon",
    "delta",
    "membership_inference_auc",
    "mia_auc",
    "reidentification_risk",
    "record_linkage_risk",
    "nearest_neighbor_disclosure_risk",
    "disclosure_risk"
]

GROUP_KEYS = ["dataset", "variant"]

# -----------------------------------------------------------------------------
# 1. NORMALIZE PRIVACY ATTACK RESULTS
# -----------------------------------------------------------------------------

if "privacy_df" not in globals():
    privacy_df = pd.DataFrame()

privacy_join = normalize_evaluation_table(
    privacy_df,
    "Privacy"
)

available_privacy_metrics = [
    col for col in PRIVACY_METRICS
    if col in privacy_join.columns
] if not privacy_join.empty else []

if not privacy_join.empty:

    for col in available_privacy_metrics:
        privacy_join[col] = pd.to_numeric(
            privacy_join[col],
            errors="coerce"
        )

# -----------------------------------------------------------------------------
# 2. BUILD COMPLETE DATASET × VARIANT GRID
# -----------------------------------------------------------------------------

expected_grid = pd.MultiIndex.from_product(
    [DATASETS, VARIANTS],
    names=GROUP_KEYS
).to_frame(index=False)

# -----------------------------------------------------------------------------
# 3. PRIVACY ATTACK COMPARISON
# -----------------------------------------------------------------------------

if not privacy_join.empty and available_privacy_metrics:

    grouped_privacy = (
        privacy_join
        .groupby(GROUP_KEYS)[available_privacy_metrics]
        .agg(["mean", "std", "count"])
    )

    grouped_privacy.columns = [
        f"{metric}_{stat}"
        for metric, stat in grouped_privacy.columns
    ]

    privacy_comparison = expected_grid.merge(
        grouped_privacy.reset_index(),
        on=GROUP_KEYS,
        how="left",
        validate="one_to_one"
    )

else:
    privacy_comparison = expected_grid.copy()

for metric in PRIVACY_METRICS:

    for stat in ["mean", "std"]:
        col = f"{metric}_{stat}"

        if col not in privacy_comparison.columns:
            privacy_comparison[col] = np.nan

    count_col = f"{metric}_count"

    if count_col not in privacy_comparison.columns:
        privacy_comparison[count_col] = 0

    privacy_comparison[count_col] = (
        privacy_comparison[count_col]
        .fillna(0)
        .astype(int)
    )

# -----------------------------------------------------------------------------
# 4. FORMAL DIFFERENTIAL PRIVACY ACCOUNTANT SUMMARY
# -----------------------------------------------------------------------------

if "dp_df" not in globals() or dp_df.empty:
    raise RuntimeError(
        "DP evidence registry is unavailable. "
        "Run Section 9 before Section 13."
    )

required_dp_columns = [
    "dataset",
    "variant",
    "achieved_epsilon",
    "delta",
    "dp_applied",
    "formal_dp_evidence_available"
]

missing_dp_columns = [
    col for col in required_dp_columns
    if col not in dp_df.columns
]

if missing_dp_columns:
    raise ValueError(
        "DP evidence registry is missing required columns: "
        f"{missing_dp_columns}"
    )

dp_input = dp_df.copy()

dp_input["dataset"] = dp_input["dataset"].astype(str)
dp_input["variant"] = dp_input["variant"].astype(str)

dp_input["achieved_epsilon"] = pd.to_numeric(
    dp_input["achieved_epsilon"],
    errors="coerce"
)

dp_input["delta"] = pd.to_numeric(
    dp_input["delta"],
    errors="coerce"
)

dp_input["dp_applied"] = (
    dp_input["dp_applied"]
    .fillna(False)
    .astype(bool)
)

dp_input["formal_dp_evidence_available"] = (
    dp_input["formal_dp_evidence_available"]
    .fillna(False)
    .astype(bool)
)

# A formal evidence record requires valid epsilon and delta values.
dp_input["valid_accountant_values"] = (
    dp_input["achieved_epsilon"].notna()
    & (dp_input["achieved_epsilon"] > 0)
    & dp_input["delta"].notna()
    & (dp_input["delta"] > 0)
    & (dp_input["delta"] < 1)
)

dp_input["validated_formal_evidence"] = (
    dp_input["dp_applied"]
    & dp_input["formal_dp_evidence_available"]
    & dp_input["valid_accountant_values"]
)

# -----------------------------------------------------------------------------
# 5. AGGREGATE FORMAL DP EVIDENCE
# -----------------------------------------------------------------------------

dp_grouped = (
    dp_input
    .groupby(GROUP_KEYS)
    .agg(
        total_runs=("dp_applied", "size"),
        dp_enabled_runs=("dp_applied", "sum"),
        mean_epsilon=("achieved_epsilon", "mean"),
        std_epsilon=("achieved_epsilon", "std"),
        mean_delta=("delta", "mean"),
        accountant_evidence=(
            "validated_formal_evidence",
            "sum"
        )
    )
    .reset_index()
)

privacy_formal_summary = expected_grid.merge(
    dp_grouped,
    on=GROUP_KEYS,
    how="left",
    validate="one_to_one"
)

count_columns = [
    "total_runs",
    "dp_enabled_runs",
    "accountant_evidence"
]

for col in count_columns:
    privacy_formal_summary[col] = (
        privacy_formal_summary[col]
        .fillna(0)
        .astype(int)
    )

# -----------------------------------------------------------------------------
# 6. SAVE PRIVACY COMPARISON ARTIFACTS
# -----------------------------------------------------------------------------

privacy_comparison_path = (
    NB17_DIRS["tables"] /
    "privacy_comparison.csv"
)

formal_summary_path = (
    NB17_DIRS["tables"] /
    "formal_privacy_comparison.csv"
)

privacy_comparison.to_csv(
    privacy_comparison_path,
    index=False
)

privacy_formal_summary.to_csv(
    formal_summary_path,
    index=False
)

# -----------------------------------------------------------------------------
# 7. VALIDATION AND STATUS
# -----------------------------------------------------------------------------

expected_groups = len(DATASETS) * len(VARIANTS)

assert len(privacy_comparison) == expected_groups
assert len(privacy_formal_summary) == expected_groups

attack_measurements = sum(
    int(privacy_comparison[f"{metric}_count"].sum())
    for metric in PRIVACY_METRICS
)

formal_evidence_count = int(
    privacy_formal_summary["accountant_evidence"].sum()
)

configured_dp_runs = int(
    privacy_formal_summary["dp_enabled_runs"].sum()
)

SECTION_13_STATUS = (
    "PASS"
    if attack_measurements > 0 and formal_evidence_count > 0
    else "BLOCKED"
)

print("PRIVACY COMPARISON")
print("=" * 80)

print("Privacy attack records:", len(privacy_join))
print("Available attack metrics:", available_privacy_metrics)
print("Valid attack metric measurements:", attack_measurements)

print("\nFormal DP accountant summary:")
print(
    privacy_formal_summary.round(7).to_string(index=False)
)

print("\nDP evidence validation:")
print("Configured DP runs:", configured_dp_runs)
print("Formal accountant evidence:", formal_evidence_count)

print("\nSaved:")
print("Privacy comparison:", privacy_comparison_path)
print("Formal DP comparison:", formal_summary_path)

print("\nSECTION 13 STATUS:", SECTION_13_STATUS)
print("=" * 80)

Privacy: empty or unavailable.
PRIVACY COMPARISON
Privacy attack records: 0
Available attack metrics: []
Valid attack metric measurements: 0

Formal DP accountant summary:
       dataset                  variant  total_runs  dp_enabled_runs  mean_epsilon  std_epsilon  mean_delta  accountant_evidence
  adult_income                 gan_only           5                0           NaN          NaN         NaN                    0
  adult_income statistical_guidance_gan           5                0           NaN          NaN         NaN                    0
  adult_income                   gan_dp           5                5           NaN          NaN         NaN                    0
  adult_income              full_sppgan           5                5           NaN          NaN         NaN                    0
bank_marketing                 gan_only           5                0           NaN          NaN         NaN                    0
bank_marketing statistical_guidance_gan           5   

In [47]:
# =============================================================================
# SECTION 14 — COMPUTATIONAL COMPARISON
# =============================================================================

import pandas as pd
import numpy as np

COMPUTATIONAL_METRICS = [
    "training_runtime_seconds",
    "generation_runtime_seconds"
]

GROUP_KEYS = ["dataset", "variant"]
RECORD_KEYS = ["dataset", "repetition_id", "variant"]

# -----------------------------------------------------------------------------
# 1. VALIDATE EXPERIMENT REGISTRY
# -----------------------------------------------------------------------------

if "registry_df" not in globals() or registry_df.empty:
    raise RuntimeError(
        "Experiment registry is unavailable. Run Section 4 first."
    )

missing_keys = [
    col for col in RECORD_KEYS
    if col not in registry_df.columns
]

if missing_keys:
    raise ValueError(
        f"Experiment registry missing required columns: {missing_keys}"
    )

# -----------------------------------------------------------------------------
# 2. EXTRACT COMPUTATIONAL METADATA
# -----------------------------------------------------------------------------

computational_df = registry_df.copy()

computational_df["dataset"] = (
    computational_df["dataset"].astype(str)
)

computational_df["variant"] = (
    computational_df["variant"].astype(str)
)

computational_df["repetition_id"] = pd.to_numeric(
    computational_df["repetition_id"],
    errors="coerce"
)

if computational_df["repetition_id"].isna().any():
    raise ValueError(
        "Invalid repetition_id values in experiment registry."
    )

if computational_df.duplicated(RECORD_KEYS).any():
    raise ValueError(
        "Duplicate experiment records found in registry."
    )

for metric in COMPUTATIONAL_METRICS:

    if metric not in computational_df.columns:
        computational_df[metric] = np.nan

    computational_df[metric] = pd.to_numeric(
        computational_df[metric],
        errors="coerce"
    )

if "model_path" not in computational_df.columns:
    computational_df["model_path"] = np.nan

# Negative runtime values are invalid measurements.
for metric in COMPUTATIONAL_METRICS:
    computational_df.loc[
        computational_df[metric] < 0,
        metric
    ] = np.nan

# Total runtime is available only when both components are measured.
computational_df["total_runtime_seconds"] = (
    computational_df["training_runtime_seconds"]
    + computational_df["generation_runtime_seconds"]
)

# -----------------------------------------------------------------------------
# 3. BUILD COMPLETE DATASET × VARIANT SUMMARY
# -----------------------------------------------------------------------------

expected_grid = pd.MultiIndex.from_product(
    [DATASETS, VARIANTS],
    names=GROUP_KEYS
).to_frame(index=False)

SUMMARY_METRICS = COMPUTATIONAL_METRICS + [
    "total_runtime_seconds"
]

summary_columns = []

for metric in SUMMARY_METRICS:
    summary_columns.extend([
        f"{metric}_mean",
        f"{metric}_std",
        f"{metric}_count"
    ])

if computational_df.empty:

    computational_summary = expected_grid.copy()

    for col in summary_columns:
        computational_summary[col] = (
            0 if col.endswith("_count") else np.nan
        )

else:

    grouped = (
        computational_df
        .groupby(GROUP_KEYS)[SUMMARY_METRICS]
        .agg(["mean", "std", "count"])
    )

    grouped.columns = [
        f"{metric}_{stat}"
        for metric, stat in grouped.columns
    ]

    computational_summary = expected_grid.merge(
        grouped.reset_index(),
        on=GROUP_KEYS,
        how="left",
        validate="one_to_one"
    )

    for col in summary_columns:
        if col not in computational_summary.columns:
            computational_summary[col] = (
                0 if col.endswith("_count") else np.nan
            )

        if col.endswith("_count"):
            computational_summary[col] = (
                computational_summary[col]
                .fillna(0)
                .astype(int)
            )

# -----------------------------------------------------------------------------
# 4. OVERALL VARIANT SUMMARY
# -----------------------------------------------------------------------------

overall_grouped = (
    computational_df
    .groupby("variant")[SUMMARY_METRICS]
    .agg(["mean", "std", "count"])
)

overall_grouped.columns = [
    f"{metric}_{stat}"
    for metric, stat in overall_grouped.columns
]

computational_overall = pd.DataFrame({
    "variant": VARIANTS
}).merge(
    overall_grouped.reset_index(),
    on="variant",
    how="left",
    validate="one_to_one"
)

for col in summary_columns:

    if col not in computational_overall.columns:
        computational_overall[col] = (
            0 if col.endswith("_count") else np.nan
        )

    if col.endswith("_count"):
        computational_overall[col] = (
            computational_overall[col]
            .fillna(0)
            .astype(int)
        )

# -----------------------------------------------------------------------------
# 5. COMPUTATIONAL COVERAGE VALIDATION
# -----------------------------------------------------------------------------

expected_records = len(DATASETS) * len(REPETITIONS) * len(VARIANTS)

assert len(computational_df) == expected_records, (
    f"Expected {expected_records} records, "
    f"found {len(computational_df)}."
)

assert len(computational_summary) == (
    len(DATASETS) * len(VARIANTS)
)

runtime_coverage = pd.DataFrame([
    {
        "metric": metric,
        "expected_records": expected_records,
        "valid_measurements": int(
            computational_df[metric].notna().sum()
        ),
        "missing_measurements": int(
            computational_df[metric].isna().sum()
        )
    }
    for metric in COMPUTATIONAL_METRICS
])

valid_runtime_measurements = int(
    runtime_coverage["valid_measurements"].sum()
)

SECTION_14_STATUS = (
    "PASS"
    if valid_runtime_measurements == expected_records * len(COMPUTATIONAL_METRICS)
    else "INCOMPLETE"
    if valid_runtime_measurements > 0
    else "BLOCKED"
)

# -----------------------------------------------------------------------------
# 6. SAVE COMPUTATIONAL ARTIFACTS
# -----------------------------------------------------------------------------

results_path = (
    NB17_DIRS["results"] /
    "computational_ablation_results.csv"
)

summary_path = (
    NB17_DIRS["tables"] /
    "computational_comparison.csv"
)

overall_path = (
    NB17_DIRS["tables"] /
    "computational_overall_comparison.csv"
)

coverage_path = (
    NB17_DIRS["validation"] /
    "computational_runtime_coverage.csv"
)

computational_df.to_csv(
    results_path,
    index=False
)

computational_summary.to_csv(
    summary_path,
    index=False
)

computational_overall.to_csv(
    overall_path,
    index=False
)

runtime_coverage.to_csv(
    coverage_path,
    index=False
)

# -----------------------------------------------------------------------------
# 7. REPORT
# -----------------------------------------------------------------------------

print("COMPUTATIONAL COMPARISON")
print("=" * 80)

print("Expected experiment records:", expected_records)
print("Computational records:", len(computational_df))
print("Valid runtime measurements:", valid_runtime_measurements)

print("\nRuntime coverage:")
print(runtime_coverage.to_string(index=False))

print("\nDataset-level computational comparison:")
print(computational_summary.round(3).to_string(index=False))

print("\nOverall variant comparison:")
print(computational_overall.round(3).to_string(index=False))

print("\nSaved:")
print("Detailed results:", results_path)
print("Dataset summary:", summary_path)
print("Overall summary:", overall_path)
print("Runtime coverage:", coverage_path)

print("\nSECTION 14 STATUS:", SECTION_14_STATUS)
print("=" * 80)

COMPUTATIONAL COMPARISON
Expected experiment records: 60
Computational records: 60
Valid runtime measurements: 0

Runtime coverage:
                    metric  expected_records  valid_measurements  missing_measurements
  training_runtime_seconds                60                   0                    60
generation_runtime_seconds                60                   0                    60

Dataset-level computational comparison:
       dataset                  variant  training_runtime_seconds_mean  training_runtime_seconds_std  training_runtime_seconds_count  generation_runtime_seconds_mean  generation_runtime_seconds_std  generation_runtime_seconds_count  total_runtime_seconds_mean  total_runtime_seconds_std  total_runtime_seconds_count
  adult_income                 gan_only                            NaN                           NaN                               0                              NaN                             NaN                                 0                   

In [49]:
# =============================================================================
# SECTION 15 — COMPONENT CONTRIBUTION ANALYSIS
# =============================================================================

import pandas as pd
import numpy as np

# -----------------------------------------------------------------------------
# 1. DEFINE IDENTIFIABLE COMPONENT COMPARISONS
# -----------------------------------------------------------------------------

# The four-variant design supports these paired comparisons.
# Dependency and distribution guidance cannot be isolated independently
# without additional ablation variants.

COMPONENT_COMPARISONS = {
    "statistical_guidance_conditional_on_dp": (
        "full_sppgan",
        "gan_dp"
    ),

    "differential_privacy_conditional_on_statistical_guidance": (
        "full_sppgan",
        "statistical_guidance_gan"
    ),

    "combined_framework": (
        "full_sppgan",
        "gan_only"
    )
}

ERROR_METRICS = [
    "mean_ks_statistic",
    "mean_wasserstein_distance",
    "mean_moment_mean_error",
    "mean_moment_std_error",
    "mean_js_distance",
    "mean_total_variation_distance",
    "correlation_frobenius_error",
    "mean_absolute_correlation_error",
    "reidentification_risk",
    "record_linkage_risk",
    "disclosure_risk",
    "nearest_neighbor_disclosure_risk",
    "membership_inference_auc",
    "mia_auc"
]

UTILITY_DIRECTION_METRICS = [
    "accuracy",
    "macro_f1",
    "weighted_f1"
]

# -----------------------------------------------------------------------------
# 2. PAIRED COMPONENT EFFECT FUNCTION
# -----------------------------------------------------------------------------

def calculate_paired_component_effect(
    df,
    metric,
    full_variant,
    ablated_variant,
    component
):

    required_columns = [
        "dataset",
        "repetition_id",
        "variant",
        metric
    ]

    if df.empty or not all(
        col in df.columns for col in required_columns
    ):
        return []

    subset = df[
        df["variant"].isin([
            full_variant,
            ablated_variant
        ])
    ].copy()

    subset[metric] = pd.to_numeric(
        subset[metric],
        errors="coerce"
    )

    records = []

    for (dataset, repetition), group in subset.groupby(
        ["dataset", "repetition_id"]
    ):

        full_values = group.loc[
            group["variant"] == full_variant,
            metric
        ].dropna()

        ablated_values = group.loc[
            group["variant"] == ablated_variant,
            metric
        ].dropna()

        # Each experiment key must have one valid value per variant.
        if len(full_values) != 1 or len(ablated_values) != 1:
            continue

        full_value = float(full_values.iloc[0])
        ablated_value = float(ablated_values.iloc[0])

        if not (
            np.isfinite(full_value)
            and np.isfinite(ablated_value)
        ):
            continue

        difference = full_value - ablated_value

        if metric in ERROR_METRICS:
            direction = (
                "Improved" if difference < 0
                else "Worsened" if difference > 0
                else "Unchanged"
            )

        elif metric in UTILITY_DIRECTION_METRICS:
            direction = (
                "Improved" if difference > 0
                else "Worsened" if difference < 0
                else "Unchanged"
            )

        else:
            direction = "Descriptive"

        records.append({
            "dataset": dataset,
            "repetition_id": repetition,
            "component": component,
            "full_variant": full_variant,
            "ablated_variant": ablated_variant,
            "metric": metric,
            "full_value": full_value,
            "ablated_value": ablated_value,
            "delta_full_minus_ablated": difference,
            "direction": direction
        })

    return records


# -----------------------------------------------------------------------------
# 3. COLLECT FIDELITY, UTILITY AND PRIVACY CONTRIBUTIONS
# -----------------------------------------------------------------------------

contribution_records = []

# Fidelity
for component, (full, ablated) in COMPONENT_COMPARISONS.items():

    for metric in FIDELITY_METRICS:

        contribution_records.extend(
            calculate_paired_component_effect(
                fidelity_df,
                metric,
                full,
                ablated,
                component
            )
        )

# Utility
if "utility_df" in globals() and not utility_df.empty:

    for component, (full, ablated) in COMPONENT_COMPARISONS.items():

        for metric in UTILITY_METRICS:

            contribution_records.extend(
                calculate_paired_component_effect(
                    utility_df,
                    metric,
                    full,
                    ablated,
                    component
                )
            )

# Empirical privacy/disclosure metrics only.
# Formal epsilon/delta evidence is not treated as an attack metric.
if "privacy_df" in globals() and not privacy_df.empty:

    for component, (full, ablated) in COMPONENT_COMPARISONS.items():

        for metric in PRIVACY_METRICS:

            if metric in ["achieved_epsilon", "delta"]:
                continue

            contribution_records.extend(
                calculate_paired_component_effect(
                    privacy_df,
                    metric,
                    full,
                    ablated,
                    component
                )
            )

# -----------------------------------------------------------------------------
# 4. CREATE STABLE OUTPUT TABLES
# -----------------------------------------------------------------------------

CONTRIBUTION_COLUMNS = [
    "dataset",
    "repetition_id",
    "component",
    "full_variant",
    "ablated_variant",
    "metric",
    "full_value",
    "ablated_value",
    "delta_full_minus_ablated",
    "direction"
]

contribution_df = pd.DataFrame(
    contribution_records,
    columns=CONTRIBUTION_COLUMNS
)

SUMMARY_COLUMNS = [
    "component",
    "metric",
    "dataset",
    "mean_component_effect",
    "std_component_effect",
    "paired_runs",
    "improved_runs",
    "worsened_runs",
    "unchanged_runs"
]

if not contribution_df.empty:

    contribution_summary = (
        contribution_df.groupby(
            ["component", "metric", "dataset"]
        )
        .agg(
            mean_component_effect=(
                "delta_full_minus_ablated", "mean"
            ),
            std_component_effect=(
                "delta_full_minus_ablated", "std"
            ),
            paired_runs=(
                "delta_full_minus_ablated", "count"
            ),
            improved_runs=(
                "direction",
                lambda x: (x == "Improved").sum()
            ),
            worsened_runs=(
                "direction",
                lambda x: (x == "Worsened").sum()
            ),
            unchanged_runs=(
                "direction",
                lambda x: (x == "Unchanged").sum()
            )
        )
        .reset_index()
    )

else:
    contribution_summary = pd.DataFrame(
        columns=SUMMARY_COLUMNS
    )

# -----------------------------------------------------------------------------
# 5. SAVE RESULTS
# -----------------------------------------------------------------------------

contribution_results_path = (
    NB17_DIRS["results"] /
    "component_contribution_results.csv"
)

contribution_summary_path = (
    NB17_DIRS["tables"] /
    "component_contribution_summary.csv"
)

contribution_df.to_csv(
    contribution_results_path,
    index=False
)

contribution_summary.to_csv(
    contribution_summary_path,
    index=False
)

# -----------------------------------------------------------------------------
# 6. VALIDATION AND STATUS
# -----------------------------------------------------------------------------

expected_comparisons = len(COMPONENT_COMPARISONS)

comparison_validation = pd.DataFrame([
    {
        "component": component,
        "full_variant": full,
        "ablated_variant": ablated,
        "paired_records": (
            int(
                contribution_df[
                    contribution_df["component"] == component
                ].shape[0]
            )
        )
    }
    for component, (full, ablated)
    in COMPONENT_COMPARISONS.items()
])

valid_paired_records = len(contribution_df)

SECTION_15_STATUS = (
    "PASS"
    if valid_paired_records > 0
    else "BLOCKED"
)

print("COMPONENT CONTRIBUTION ANALYSIS")
print("=" * 80)

print("Identifiable component comparisons:", expected_comparisons)
print("Valid paired contribution records:", valid_paired_records)

print("\nComparison validation:")
print(comparison_validation.to_string(index=False))

print("\nComponent contribution summary:")
if contribution_summary.empty:
    print("No valid paired component results available.")
else:
    print(
        contribution_summary.round(6)
        .to_string(index=False)
    )

print("\nSaved:")
print("Detailed results:", contribution_results_path)
print("Summary:", contribution_summary_path)

print("\nSECTION 15 STATUS:", SECTION_15_STATUS)

print("\nMETHODOLOGICAL SCOPE:")
print(
    "- Full SPP-GAN vs GAN-DP estimates statistical-guidance "
    "contribution conditional on DP."
)
print(
    "- Full SPP-GAN vs statistical-guidance GAN estimates DP "
    "contribution conditional on statistical guidance."
)
print(
    "- Full SPP-GAN vs GAN-only estimates the combined framework effect."
)
print(
    "- Dependency and distribution guidance are not independently "
    "identifiable from the current four-variant design."
)

print("=" * 80)

COMPONENT CONTRIBUTION ANALYSIS
Identifiable component comparisons: 3
Valid paired contribution records: 0

Comparison validation:
                                               component full_variant          ablated_variant  paired_records
                  statistical_guidance_conditional_on_dp  full_sppgan                   gan_dp               0
differential_privacy_conditional_on_statistical_guidance  full_sppgan statistical_guidance_gan               0
                                      combined_framework  full_sppgan                 gan_only               0

Component contribution summary:
No valid paired component results available.

Saved:
Detailed results: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_17/results/component_contribution_results.csv
Summary: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_17/tables/component_contribution_summary.csv

SECTION 15 STATUS: BLOCKED

METHODOLOGICAL SCOPE:
- Full SPP-GAN vs GAN-DP estimates s

In [51]:
# =============================================================================
# SECTION 16 — SAVE ABLATION RESULTS
# =============================================================================

import hashlib
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datetime import datetime, timezone
from pathlib import Path

# -----------------------------------------------------------------------------
# 1. VALIDATE OUTPUT DIRECTORIES
# -----------------------------------------------------------------------------

for directory in NB17_DIRS.values():
    Path(directory).mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# 2. SAVE CONSOLIDATED EXPERIMENT RESULTS
# -----------------------------------------------------------------------------

ARTIFACTS_TO_SAVE = {
    "ablation_results.csv": evaluation_df,
    "fidelity_ablation_results.csv": fidelity_df,
    "dp_ablation_results.csv": dp_df,
    "computational_ablation_results.csv": computational_df,
    "component_contribution_results.csv": contribution_df
}

saved_artifacts = []

for filename, dataframe in ARTIFACTS_TO_SAVE.items():

    if not isinstance(dataframe, pd.DataFrame):
        raise TypeError(
            f"{filename}: expected a pandas DataFrame."
        )

    output_path = (
        NB17_DIRS["results"] / filename
    )

    dataframe.to_csv(
        output_path,
        index=False
    )

    saved_artifacts.append(output_path)

    print(
        f"Saved: {filename} | Rows: {len(dataframe)}"
    )

# -----------------------------------------------------------------------------
# 3. GENERATE FIDELITY FIGURES
# -----------------------------------------------------------------------------

fidelity_figures_created = 0

if not fidelity_df.empty:

    for metric in available_fidelity_metrics:

        if metric not in fidelity_df.columns:
            continue

        plot_data = fidelity_df.copy()

        plot_data[metric] = pd.to_numeric(
            plot_data[metric],
            errors="coerce"
        )

        plot_data = (
            plot_data.groupby("variant")[metric]
            .mean()
            .dropna()
        )

        if plot_data.empty:
            continue

        fig, ax = plt.subplots(figsize=(10, 5))

        plot_data.plot(
            kind="bar",
            ax=ax
        )

        ax.set_title(
            f"Ablation Comparison — {metric}"
        )
        ax.set_xlabel("Ablation Variant")
        ax.set_ylabel(metric)
        ax.tick_params(axis="x", rotation=25)

        fig.tight_layout()

        safe_name = metric.replace("/", "_")

        figure_path = (
            NB17_DIRS["figures"] /
            f"ablation_{safe_name}.png"
        )

        fig.savefig(
            figure_path,
            dpi=300,
            bbox_inches="tight"
        )

        plt.close(fig)

        fidelity_figures_created += 1

# -----------------------------------------------------------------------------
# 4. GENERATE COMPONENT CONTRIBUTION FIGURES
# -----------------------------------------------------------------------------

contribution_figures_created = 0

if not contribution_summary.empty:

    selected = contribution_summary[
        contribution_summary["metric"].isin(
            FIDELITY_METRICS[:4]
        )
    ]

    for metric in selected["metric"].dropna().unique():

        plot_data = selected[
            selected["metric"] == metric
        ]

        if plot_data.empty:
            continue

        pivot = plot_data.pivot_table(
            index="component",
            columns="dataset",
            values="mean_component_effect",
            aggfunc="mean"
        )

        if pivot.empty or pivot.isna().all().all():
            continue

        fig, ax = plt.subplots(figsize=(11, 6))

        pivot.plot(
            kind="bar",
            ax=ax
        )

        ax.set_title(
            f"Component Contribution — {metric}"
        )

        ax.set_ylabel(
            "Full SPP-GAN minus Ablated Variant"
        )

        ax.tick_params(axis="x", rotation=25)

        fig.tight_layout()

        figure_path = (
            NB17_DIRS["figures"] /
            f"component_contribution_{metric}.png"
        )

        fig.savefig(
            figure_path,
            dpi=300,
            bbox_inches="tight"
        )

        plt.close(fig)

        contribution_figures_created += 1

# -----------------------------------------------------------------------------
# 5. SAVE EXPERIMENT CONFIGURATION
# -----------------------------------------------------------------------------

configuration = {
    "project_root": str(PROJECT_ROOT),
    "notebook": "Notebook 17",
    "experiment": "Ablation Study",
    "configuration": ABLATION_CONFIG,
    "variants": VARIANT_REGISTRY,
    "master_seed": MASTER_SEED,
    "repetitions": list(REPETITIONS),
    "expected_experiment_records": (
        len(DATASETS) * len(REPETITIONS) * len(VARIANTS)
    ),
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

configuration_path = (
    NB17_DIRS["metadata"] /
    "ablation_experiment_configuration.json"
)

with open(configuration_path, "w", encoding="utf-8") as file:
    json.dump(
        configuration,
        file,
        indent=4,
        default=str
    )

# -----------------------------------------------------------------------------
# 6. GENERATE SHA-256 INTEGRITY MANIFEST
# -----------------------------------------------------------------------------

def calculate_sha256(path):

    digest = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


manifest_path = (
    NB17_DIRS["manifests"] /
    "artifact_integrity_manifest.csv"
)

artifact_manifest = []

# Hash every existing artifact after all outputs and configuration
# have been written. Exclude only the manifest being generated.
for directory in NB17_DIRS.values():

    for path in Path(directory).rglob("*"):

        if not path.is_file():
            continue

        if path.resolve() == manifest_path.resolve():
            continue

        artifact_manifest.append({
            "artifact_path": str(path),
            "artifact_name": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": calculate_sha256(path)
        })


MANIFEST_COLUMNS = [
    "artifact_path",
    "artifact_name",
    "size_bytes",
    "sha256"
]

artifact_manifest_df = pd.DataFrame(
    artifact_manifest,
    columns=MANIFEST_COLUMNS
)

artifact_manifest_df.to_csv(
    manifest_path,
    index=False
)

# -----------------------------------------------------------------------------
# 7. VERIFY SAVED ARTIFACTS
# -----------------------------------------------------------------------------

missing_artifacts = [
    str(path)
    for path in saved_artifacts + [configuration_path, manifest_path]
    if not path.exists()
]

manifest_validation = (
    not artifact_manifest_df.empty
    and artifact_manifest_df["sha256"].notna().all()
    and artifact_manifest_df["sha256"].str.len().eq(64).all()
)

save_validation = (
    len(missing_artifacts) == 0
    and manifest_validation
)

# -----------------------------------------------------------------------------
# 8. FINAL SECTION REPORT
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("SECTION 16 — ABLATION ARTIFACTS SAVED")
print("=" * 80)

for name, directory in NB17_DIRS.items():

    file_count = sum(
        1 for path in Path(directory).rglob("*")
        if path.is_file()
    )

    print(
        f"{name:15s}: {file_count} files"
    )

print("\nFidelity figures created:", fidelity_figures_created)
print(
    "Component figures created:",
    contribution_figures_created
)

print(
    "Artifact integrity records:",
    len(artifact_manifest_df)
)

print("Configuration saved:", configuration_path)
print("Integrity manifest saved:", manifest_path)

print("Missing required artifacts:", len(missing_artifacts))

if missing_artifacts:
    for path in missing_artifacts:
        print("MISSING:", path)

SECTION_16_STATUS = (
    "PASS" if save_validation else "INCOMPLETE"
)

print("\nSECTION 16 STATUS:", SECTION_16_STATUS)
print("=" * 80)

Saved: ablation_results.csv | Rows: 60
Saved: fidelity_ablation_results.csv | Rows: 0
Saved: dp_ablation_results.csv | Rows: 60
Saved: computational_ablation_results.csv | Rows: 60
Saved: component_contribution_results.csv | Rows: 0

SECTION 16 — ABLATION ARTIFACTS SAVED
results        : 8 files
tables         : 12 files
metadata       : 3 files
validation     : 4 files
figures        : 0 files
manifests      : 3 files
logs           : 0 files
registries     : 2 files

Fidelity figures created: 0
Component figures created: 0
Artifact integrity records: 31
Configuration saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_17/metadata/ablation_experiment_configuration.json
Integrity manifest saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_17/manifests/artifact_integrity_manifest.csv
Missing required artifacts: 0

SECTION 16 STATUS: PASS


In [53]:
# =============================================================================
# SECTION 17 — VALIDATE ABLATION COVERAGE
# =============================================================================

import pandas as pd
import numpy as np
import hashlib

from pathlib import Path

# -----------------------------------------------------------------------------
# 1. INITIALIZE VALIDATION
# -----------------------------------------------------------------------------

validation_checks = []

def add_check(name, condition, details=""):

    validation_checks.append({
        "validation": name,
        "status": "PASS" if bool(condition) else "FAIL",
        "details": str(details)
    })


# -----------------------------------------------------------------------------
# 2. EXPECTED EXPERIMENT GRID
# -----------------------------------------------------------------------------

EXPECTED_RECORDS = (
    len(DATASETS) *
    len(REPETITIONS) *
    len(VARIANTS)
)

JOIN_KEYS = [
    "dataset",
    "repetition_id",
    "variant"
]

expected_grid = pd.MultiIndex.from_product(
    [DATASETS, REPETITIONS, VARIANTS],
    names=JOIN_KEYS
).to_frame(index=False)

registry_validation = registry_df.copy()

for col in ["dataset", "variant"]:
    registry_validation[col] = (
        registry_validation[col].astype(str)
    )

registry_validation["repetition_id"] = pd.to_numeric(
    registry_validation["repetition_id"],
    errors="coerce"
)

registry_keys = registry_validation[JOIN_KEYS].dropna()

actual_key_set = set(
    map(tuple, registry_keys.to_numpy())
)

expected_key_set = set(
    map(tuple, expected_grid[JOIN_KEYS].to_numpy())
)

missing_keys_current = expected_key_set - actual_key_set
unexpected_keys_current = actual_key_set - expected_key_set

duplicate_keys = int(
    registry_validation.duplicated(JOIN_KEYS).sum()
)

# -----------------------------------------------------------------------------
# 3. REGISTRY COVERAGE VALIDATION
# -----------------------------------------------------------------------------

add_check(
    "Expected registry records",
    len(registry_df) == EXPECTED_RECORDS,
    f"{len(registry_df)} / {EXPECTED_RECORDS}"
)

add_check(
    "Complete dataset-repetition-variant coverage",
    len(missing_keys_current) == 0
    and len(unexpected_keys_current) == 0,
    (
        f"Missing={len(missing_keys_current)}, "
        f"Unexpected={len(unexpected_keys_current)}"
    )
)

add_check(
    "Unique experimental identifiers",
    duplicate_keys == 0,
    f"Duplicate records={duplicate_keys}"
)

# -----------------------------------------------------------------------------
# 4. SYNTHETIC OUTPUT VALIDATION
# -----------------------------------------------------------------------------

synthetic_column_available = (
    "synthetic_exists" in registry_validation.columns
)

if synthetic_column_available:

    synthetic_exists = (
        registry_validation["synthetic_exists"]
        .fillna(False)
        .astype(bool)
    )

    synthetic_count = int(synthetic_exists.sum())

else:
    synthetic_count = 0

add_check(
    "Synthetic output availability",
    synthetic_column_available
    and synthetic_count == EXPECTED_RECORDS,
    f"{synthetic_count} / {EXPECTED_RECORDS}"
)

# -----------------------------------------------------------------------------
# 5. FIDELITY VALIDATION
# -----------------------------------------------------------------------------

fidelity_count = (
    len(fidelity_df)
    if isinstance(fidelity_df, pd.DataFrame)
    else 0
)

fidelity_variants = (
    set(fidelity_df["variant"].dropna().astype(str).unique())
    if fidelity_count > 0 and "variant" in fidelity_df.columns
    else set()
)

add_check(
    "Fidelity result availability",
    fidelity_count == EXPECTED_RECORDS,
    f"{fidelity_count} / {EXPECTED_RECORDS}"
)

add_check(
    "Fidelity variant coverage",
    fidelity_variants == set(VARIANTS),
    f"Available={sorted(fidelity_variants)}"
)

# -----------------------------------------------------------------------------
# 6. UTILITY VALIDATION
# -----------------------------------------------------------------------------

utility_count = (
    len(utility_df)
    if isinstance(utility_df, pd.DataFrame)
    else 0
)

add_check(
    "TSTR utility availability",
    utility_count == EXPECTED_RECORDS,
    f"{utility_count} / {EXPECTED_RECORDS}"
)

# -----------------------------------------------------------------------------
# 7. PRIVACY VALIDATION
# -----------------------------------------------------------------------------

privacy_count = (
    len(privacy_df)
    if isinstance(privacy_df, pd.DataFrame)
    else 0
)

add_check(
    "Privacy artifact availability",
    privacy_count == EXPECTED_RECORDS,
    f"{privacy_count} / {EXPECTED_RECORDS}"
)

# -----------------------------------------------------------------------------
# 8. FORMAL DP ACCOUNTANT EVIDENCE
# -----------------------------------------------------------------------------

expected_dp_records = (
    len(DATASETS) *
    len(REPETITIONS) *
    2
)

if (
    isinstance(dp_df, pd.DataFrame)
    and "formal_dp_evidence_available" in dp_df.columns
):

    dp_evidence_count = int(
        dp_df["formal_dp_evidence_available"]
        .fillna(False)
        .astype(bool)
        .sum()
    )

else:
    dp_evidence_count = 0

add_check(
    "DP accountant evidence",
    dp_evidence_count == expected_dp_records,
    f"{dp_evidence_count} / {expected_dp_records}"
)

# -----------------------------------------------------------------------------
# 9. COMPUTATIONAL METADATA
# -----------------------------------------------------------------------------

runtime_metrics = [
    "training_runtime_seconds",
    "generation_runtime_seconds"
]

runtime_coverage = {}

for metric in runtime_metrics:

    if (
        isinstance(computational_df, pd.DataFrame)
        and metric in computational_df.columns
    ):

        runtime_coverage[metric] = int(
            pd.to_numeric(
                computational_df[metric],
                errors="coerce"
            ).notna().sum()
        )

    else:
        runtime_coverage[metric] = 0

runtime_complete = all(
    count == EXPECTED_RECORDS
    for count in runtime_coverage.values()
)

add_check(
    "Training runtime metadata",
    runtime_coverage["training_runtime_seconds"] == EXPECTED_RECORDS,
    (
        f"{runtime_coverage['training_runtime_seconds']} / "
        f"{EXPECTED_RECORDS}"
    )
)

add_check(
    "Generation runtime metadata",
    runtime_coverage["generation_runtime_seconds"] == EXPECTED_RECORDS,
    (
        f"{runtime_coverage['generation_runtime_seconds']} / "
        f"{EXPECTED_RECORDS}"
    )
)

# -----------------------------------------------------------------------------
# 10. COMPONENT CONTRIBUTION VALIDATION
# -----------------------------------------------------------------------------

contribution_count = (
    len(contribution_df)
    if isinstance(contribution_df, pd.DataFrame)
    else 0
)

add_check(
    "Component contribution results",
    contribution_count > 0,
    f"{contribution_count} paired records"
)

# -----------------------------------------------------------------------------
# 11. ARTIFACT INTEGRITY VALIDATION
# -----------------------------------------------------------------------------

manifest_path = (
    NB17_DIRS["manifests"] /
    "artifact_integrity_manifest.csv"
)

manifest_exists = manifest_path.exists()
manifest_valid = False
manifest_records = 0

if manifest_exists:

    try:
        manifest_df = pd.read_csv(manifest_path)

        required_manifest_columns = [
            "artifact_path",
            "artifact_name",
            "size_bytes",
            "sha256"
        ]

        if all(
            col in manifest_df.columns
            for col in required_manifest_columns
        ):

            manifest_records = len(manifest_df)

            manifest_valid = (
                not manifest_df.empty
                and manifest_df["sha256"].notna().all()
                and manifest_df["sha256"].astype(str)
                .str.fullmatch(r"[a-fA-F0-9]{64}").all()
            )

    except Exception:
        manifest_valid = False

add_check(
    "Artifact integrity manifest",
    manifest_exists and manifest_valid,
    f"Manifest records={manifest_records}"
)

# -----------------------------------------------------------------------------
# 12. SAVE VALIDATION REPORT
# -----------------------------------------------------------------------------

validation_df = pd.DataFrame(
    validation_checks
)

validation_report_path = (
    NB17_DIRS["validation"] /
    "ablation_validation_report.csv"
)

validation_df.to_csv(
    validation_report_path,
    index=False
)

passed = int(
    (validation_df["status"] == "PASS").sum()
)

failed = int(
    (validation_df["status"] == "FAIL").sum()
)

total_checks = len(validation_df)

# -----------------------------------------------------------------------------
# 13. OVERALL VALIDATION STATUS
# -----------------------------------------------------------------------------

SECTION_17_STATUS = (
    "PASS"
    if failed == 0
    else "BLOCKED"
)

# -----------------------------------------------------------------------------
# 14. SAVE VALIDATION SUMMARY
# -----------------------------------------------------------------------------

validation_summary = {
    "notebook": "Notebook 17",
    "section": "Section 17 — Validate Ablation Coverage",
    "expected_experiment_records": EXPECTED_RECORDS,
    "registry_records": len(registry_df),
    "synthetic_outputs_available": synthetic_count,
    "fidelity_records": fidelity_count,
    "utility_records": utility_count,
    "privacy_records": privacy_count,
    "dp_accountant_evidence": dp_evidence_count,
    "training_runtime_records": runtime_coverage[
        "training_runtime_seconds"
    ],
    "generation_runtime_records": runtime_coverage[
        "generation_runtime_seconds"
    ],
    "component_contribution_records": contribution_count,
    "passed_checks": passed,
    "failed_checks": failed,
    "total_checks": total_checks,
    "section_status": SECTION_17_STATUS
}

summary_path = (
    NB17_DIRS["validation"] /
    "ablation_validation_summary.json"
)

with open(summary_path, "w", encoding="utf-8") as file:
    json.dump(
        validation_summary,
        file,
        indent=4
    )

# -----------------------------------------------------------------------------
# 15. FINAL REPORT
# -----------------------------------------------------------------------------

print("=" * 80)
print("ABLATION VALIDATION REPORT")
print("=" * 80)

print(validation_df.to_string(index=False))

print("\nPassed checks:", passed)
print("Failed checks:", failed)
print("Total checks:", total_checks)

print("\nValidation summary:")
for key, value in validation_summary.items():
    print(f"{key}: {value}")

print("\nSaved:")
print("Validation report:", validation_report_path)
print("Validation summary:", summary_path)

print("\nSECTION 17 STATUS:", SECTION_17_STATUS)
print("=" * 80)

ABLATION VALIDATION REPORT
                                  validation status                 details
                   Expected registry records   PASS                 60 / 60
Complete dataset-repetition-variant coverage   PASS Missing=0, Unexpected=0
             Unique experimental identifiers   PASS     Duplicate records=0
               Synthetic output availability   FAIL                  0 / 60
                Fidelity result availability   FAIL                  0 / 60
                   Fidelity variant coverage   FAIL            Available=[]
                   TSTR utility availability   FAIL                  0 / 60
               Privacy artifact availability   FAIL                  0 / 60
                      DP accountant evidence   FAIL                  0 / 30
                   Training runtime metadata   FAIL                  0 / 60
                 Generation runtime metadata   FAIL                  0 / 60
              Component contribution results   FAIL        0 

In [55]:
# =============================================================================
# SECTION 18 — COMPLETION SUMMARY
# =============================================================================

import json
import gc
import pandas as pd
import numpy as np

from datetime import datetime, timezone
from pathlib import Path

# -----------------------------------------------------------------------------
# 1. COMPLETION METADATA
# -----------------------------------------------------------------------------

completion_time = datetime.now(
    timezone.utc
).isoformat()

EXPECTED_RECORDS = (
    len(DATASETS) *
    len(REPETITIONS) *
    len(VARIANTS)
)

# -----------------------------------------------------------------------------
# 2. COLLECT EXPERIMENTAL COUNTS
# -----------------------------------------------------------------------------

registry_records = (
    len(registry_df)
    if isinstance(registry_df, pd.DataFrame)
    else 0
)

synthetic_outputs = (
    int(
        registry_df["synthetic_exists"]
        .fillna(False)
        .astype(bool)
        .sum()
    )
    if (
        isinstance(registry_df, pd.DataFrame)
        and "synthetic_exists" in registry_df.columns
    )
    else 0
)

fidelity_records = len(fidelity_df)
utility_records = len(utility_df)
privacy_records = len(privacy_df)
contribution_records = len(contribution_df)

formal_dp_evidence = int(
    dp_evidence_count
    if "dp_evidence_count" in globals()
    else 0
)

# -----------------------------------------------------------------------------
# 3. DETERMINE FINAL RESEARCH STATUS
# -----------------------------------------------------------------------------

validation_failures = (
    int(failed)
    if "failed" in globals()
    else 0
)

validation_passes = (
    int(passed)
    if "passed" in globals()
    else 0
)

validation_total = (
    len(validation_df)
    if "validation_df" in globals()
    else 0
)

if validation_failures == 0 and validation_total > 0:
    completion_status = "COMPLETED"
elif registry_records == EXPECTED_RECORDS:
    completion_status = "COMPLETED_WITH_MISSING_EVIDENCE"
else:
    completion_status = "INCOMPLETE"

research_evaluation_complete = (
    completion_status == "COMPLETED"
    and synthetic_outputs == EXPECTED_RECORDS
    and fidelity_records == EXPECTED_RECORDS
    and utility_records == EXPECTED_RECORDS
    and privacy_records == EXPECTED_RECORDS
    and formal_dp_evidence == len(DATASETS) * len(REPETITIONS) * 2
)

# -----------------------------------------------------------------------------
# 4. BUILD COMPLETION SUMMARY
# -----------------------------------------------------------------------------

completion_summary = {

    "notebook": "Notebook 17 — Ablation Study",

    "research_framework": "SPP-GAN",

    "completion_timestamp_utc": completion_time,

    "project_root": str(PROJECT_ROOT),

    "output_root": str(NB17_ROOT),

    "experimental_design": {
        "datasets": list(DATASETS),
        "repetitions": list(REPETITIONS),
        "variants": list(VARIANTS),
        "expected_records": EXPECTED_RECORDS
    },

    "experimental_coverage": {
        "registry_records": registry_records,
        "synthetic_outputs_available": synthetic_outputs,
        "synthetic_outputs_missing": (
            EXPECTED_RECORDS - synthetic_outputs
        ),
        "fidelity_records": fidelity_records,
        "utility_records": utility_records,
        "privacy_records": privacy_records,
        "component_contribution_records": contribution_records
    },

    "computational_evidence": {
        "training_runtime_records": int(
            computational_df[
                "training_runtime_seconds"
            ].notna().sum()
        ) if "training_runtime_seconds" in computational_df.columns else 0,

        "generation_runtime_records": int(
            computational_df[
                "generation_runtime_seconds"
            ].notna().sum()
        ) if "generation_runtime_seconds" in computational_df.columns else 0
    },

    "privacy_reporting": {
        "accountant_configuration": "RDP",
        "private_component_configuration": "Discriminator / Critic",
        "target_epsilon": 5.0,
        "formal_dp_evidence_records": formal_dp_evidence,
        "achieved_privacy_guarantee_verified": (
            formal_dp_evidence > 0
        ),
        "end_to_end_dp_claim_automatically_assumed": False
    },

    "validation": {
        "passed": validation_passes,
        "failed": validation_failures,
        "total": validation_total,
        "validation_status": (
            "PASS" if validation_failures == 0
            else "BLOCKED"
        )
    },

    "results": {
        "fidelity_evaluated": fidelity_records > 0,
        "utility_evaluated": utility_records > 0,
        "privacy_evaluated": privacy_records > 0,
        "component_contribution_evaluated": (
            contribution_records > 0
        )
    },

    "execution": {
        "notebook_execution_finished": True,
        "research_evaluation_complete": research_evaluation_complete,
        "status": completion_status
    }
}

# -----------------------------------------------------------------------------
# 5. SAVE JSON COMPLETION SUMMARY
# -----------------------------------------------------------------------------

summary_path = (
    NB17_DIRS["metadata"] /
    "notebook_17_completion_summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        completion_summary,
        file,
        indent=4,
        default=str
    )

# -----------------------------------------------------------------------------
# 6. SAVE READABLE COMPLETION REPORT
# -----------------------------------------------------------------------------

report_path = (
    NB17_DIRS["logs"] /
    "notebook_17_completion_report.txt"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as file:

    file.write(
        "NOTEBOOK 17 — ABLATION STUDY\n"
        + "=" * 80 + "\n\n"
    )

    file.write(
        json.dumps(
            completion_summary,
            indent=4,
            default=str
        )
    )

    file.write("\n")

# -----------------------------------------------------------------------------
# 7. FINAL COMPLETION REPORT
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("NOTEBOOK 17 — COMPLETION SUMMARY")
print("=" * 80)

print("Status:", completion_status)
print("Execution finished:", True)
print("Research evaluation complete:", research_evaluation_complete)

print("\nExperimental coverage:")
print("Expected records:", EXPECTED_RECORDS)
print("Registry records:", registry_records)
print("Synthetic outputs:", synthetic_outputs)
print("Fidelity results:", fidelity_records)
print("Utility results:", utility_records)
print("Privacy results:", privacy_records)
print("Contribution results:", contribution_records)

print("\nPrivacy evidence:")
print("Formal DP accountant evidence:", formal_dp_evidence)
print("Achieved privacy guarantee verified:",
      formal_dp_evidence > 0)

print("\nValidation:")
print("Passed:", validation_passes)
print("Failed:", validation_failures)
print("Total:", validation_total)

print("\nProject:", PROJECT_ROOT)
print("Notebook output:", NB17_ROOT)

print("\nCompletion summary:", summary_path)
print("Completion report:", report_path)

print("=" * 80)
print("NOTEBOOK 17 EXECUTION FINISHED")
print("=" * 80)

gc.collect()


NOTEBOOK 17 — COMPLETION SUMMARY
Status: COMPLETED_WITH_MISSING_EVIDENCE
Execution finished: True
Research evaluation complete: False

Experimental coverage:
Expected records: 60
Registry records: 60
Synthetic outputs: 0
Fidelity results: 0
Utility results: 0
Privacy results: 0
Contribution results: 0

Privacy evidence:
Formal DP accountant evidence: 0
Achieved privacy guarantee verified: False

Validation:
Passed: 4
Failed: 9
Total: 13

Project: /content/drive/MyDrive/SPP_GAN_Research
Notebook output: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_17

Completion summary: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_17/metadata/notebook_17_completion_summary.json
Completion report: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_17/logs/notebook_17_completion_report.txt
NOTEBOOK 17 EXECUTION FINISHED


31